Data Source:

The core data source for this project is a list of deaths in National Park units as reported by the US National Park Service in a file downloaded from this page: https://www.nps.gov/aboutus/mortality-data.htm. I'll start with required imports and loading that dataset to a pandas dataframe.

In [140]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import geopandas as gpd

import requests, os, time, lxml, re, unicodedata, sqlite3

from dotenv import load_dotenv

from bs4 import BeautifulSoup

In [141]:
API_KEY = os.getenv("NPS_API_KEY")

print(API_KEY is not None)

True


I'm going to create a function that will normalize columns that are strings.

In [142]:
def clean(col):
    #Verify is string
    col = col.astype("string")

    #Strip leading/trailing whitespace and lowercase
    col = col.str.strip().str.lower()

    #Normalize unicode (remove accents/diacritics: haleakala, honokohau, hawaii, etc.)
    col = col.map(lambda x: unicodedata.normalize("NFKD", x) if pd.notna(x) else x)
    col = col.str.encode("ascii", "ignore").str.decode("ascii")

    #Drop curly apostrophes/okinas
    col = col.str.replace(r"[’ʻ`]", "", regex=True)

    # Normalize different dash types
    col = col.str.replace(r"[-–—]", "-", regex=True)

    # Normalize "&" to "and"
    col = col.str.replace(r"\s*&\s*", " and ", regex=True)

    # Normalize "st." → "st"
    col = col.str.replace(r"\bst\.\s", "st ", regex=True)

    # Collapse multiple spaces to a single space and strip again
    col = col.str.replace(r"\s+", " ", regex=True).str.strip()

    # Treat empty strings as missing
    col = col.replace("", pd.NA)

    return col

In [143]:
df = pd.ExcelFile(r"../data/NPS-Mortality-Data-CY2007-to-CY2024-Released-August-2024.xlsx")
df = df.parse(sheet_name="CY2007-Present Q2")
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity
0,2007-01-01,Glen Canyon National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,65+,Not Reported
1,2007-01-22,Golden Gate National Recreation Area,Drowning,Drowning,Unintentional,Fatal injury,Male,Not Reported,Vessel Related
2,2007-01-22,Golden Gate National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,Not Reported,Vessel Related
3,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,15-24,Driving
4,2007-01-29,Natchez Trace Parkway,Motor Vehicle Crash,Motor Vehicle Crash,Unintentional,Fatal injury,Female,45-54,Driving


In [144]:
df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death',
       'Cause of Death Group \n(Used in the NPS Mortality Dashboard) ',
       'Intent', 'Outcome', 'Sex', 'Age Range', 'Activity'],
      dtype='object')

I need to normalize the name, but I want to create a duplicate column that will retain the original information including all special characters.

In [145]:
df['original_name'] = df['Park Name'].copy()

df.head(1)

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity,original_name
0,2007-01-01,Glen Canyon National Recreation Area,Undetermined,Undetermined,Undetermined,Fatal injury,Male,65+,Not Reported,Glen Canyon National Recreation Area


In [146]:
cols = ["Park Name", 
        "Intent", 
        "Outcome", 
        "Cause of Death", 
        "Cause of Death Group \n(Used in the NPS Mortality Dashboard) ", 
        "Sex", 
        "Age Range",
        "Activity"]

df[cols] = df[cols].apply(clean)

In [147]:
df.head(1)

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Outcome,Sex,Age Range,Activity,original_name
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,fatal injury,male,65+,not reported,Glen Canyon National Recreation Area


In [148]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4635 entries, 0 to 4634
Data columns (total 10 columns):
 #   Column                                                        Non-Null Count  Dtype         
---  ------                                                        --------------  -----         
 0   Incident Date                                                 4635 non-null   datetime64[ns]
 1   Park Name                                                     4635 non-null   object        
 2   Cause of Death                                                4635 non-null   object        
 3   Cause of Death Group 
(Used in the NPS Mortality Dashboard)   4635 non-null   object        
 4   Intent                                                        4635 non-null   object        
 5   Outcome                                                       4635 non-null   object        
 6   Sex                                                           4635 non-null   object        
 7   Age Ra

In [149]:
df.isna().sum()

Incident Date                                                     0
Park Name                                                         0
Cause of Death                                                    0
Cause of Death Group \n(Used in the NPS Mortality Dashboard)      0
Intent                                                            0
Outcome                                                           0
Sex                                                               0
Age Range                                                        12
Activity                                                          0
original_name                                                     0
dtype: int64

I'm going to look more closely at the "Cause of death" and "Cause of Death Group \n..." columns to see if this data is redundant, or if I need to use both in my analysis.

Check if all values in all rows and columns are the same.

In [150]:
(df["Cause of Death"] == df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]).all()

np.False_

Since the are not duplicates, I'm going to build a mask to count how many mismatches are in the data.

In [151]:
mask_mismatch = df["Cause of Death"] != df["Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]
print("Total rows:", len(df))
print("Total mismatches:", mask_mismatch.sum())

Total rows: 4635
Total mismatches: 494


In [152]:
df.loc[mask_mismatch, ["Cause of Death", "Cause of Death Group \n(Used in the NPS Mortality Dashboard) "]].head()

,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard)
8,avalanche,environmental
9,vessel incident,other transportation
46,vessel incident,other transportation
48,bicycle only crash,other transportation
56,poisoning - drugs,poisoning


First, I'll focus on "Cause of Death" fields.

In [153]:
df.groupby("Cause of Death").size()

Cause of Death
aircraft incident                          77
altitude                                    1
asphyxiation                                7
avalanche                                  41
bicycle only crash                         23
collapsing earth/sand                       2
drowning                                  864
electrocution                               4
fall                                      505
falling ice                                 2
falling tree/branch                        21
fire/burn                                   5
firearm                                     5
flash flood                                14
homicide                                   51
horseback riding incident                   4
hyperthermia                               83
hypothermia                                46
legal intervention                          2
lightning strike                           10
medical - during physical activity        318
medical - not durin

In [154]:
df["Cause of Death"].nunique()

42

In [155]:
print(sorted(df["Cause of Death"].unique()))

['aircraft incident', 'altitude', 'asphyxiation', 'avalanche', 'bicycle only crash', 'collapsing earth/sand', 'drowning', 'electrocution', 'fall', 'falling ice', 'falling tree/branch', 'fire/burn', 'firearm', 'flash flood', 'homicide', 'horseback riding incident', 'hyperthermia', 'hypothermia', 'legal intervention', 'lightning strike', 'medical - during physical activity', 'medical - not during physical activity', 'medical - unknown', 'motor vehicle crash', 'other', 'poisoning - alcohol', 'poisoning - carbon monoxide', 'poisoning - drugs', 'poisoning - other', 'rockfall', 'skateboard incident', 'skiing incident', 'snowboard incident', 'snowmobile incident', 'struck by/against', 'suicide', 'train crash', 'trolly crash', 'undetermined', 'vessel incident', 'water diving incident', 'widlife incident']


Now I'll look at age range fields.

In [156]:
print(df["Age Range"].unique())

['65+' 'not reported' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14' <NA>
 '45 - 54' '35 - 44' 'unintentional' '0 - 14']


In [157]:
#combine unknown field values
unknown_age = {"not reported", "unintentional", "unknown", None, np.nan}

df["age_range_clean"] = df["Age Range"].replace(list(unknown_age), "Unknown")

#remove extra spaces to combine age groups
df["age_range_clean"] = (
    df["age_range_clean"]
    .str.replace(r"\s*-\s*", "-", regex=True)  # normalize dashes
    .str.strip()
)

print(df["age_range_clean"].value_counts())

age_range_clean
65+        827
55-64      676
Unknown    643
45-54      627
25-34      616
15-24      594
35-44      542
0-14       110
Name: count, dtype: int64


In [158]:
#copy helper column and drop helper column
df["Age Range"] = df["age_range_clean"]

df = df.drop(columns=["age_range_clean"])

In [159]:
print(df["Age Range"].unique())

['65+' 'Unknown' '15-24' '45-54' '25-34' '55-64' '35-44' '0-14']


In [160]:
ranges = df["Age Range"].str.extract(r"(?P<min>\d+)-(?P<max>\d+)")
df["Age Range Min"] = ranges["min"].astype(float)
df["Age Range Max"] = ranges["max"].astype(float)

mask_plus = df["Age Range"].str.endswith("+", na=False)
df.loc[mask_plus, "Age Range Min"] = df.loc[mask_plus, "Age Range"].str[:-1].astype(float)
df.loc[mask_plus, "Age Range Max"] = np.nan 

In [161]:
df.dtypes

Incident Date                                                    datetime64[ns]
Park Name                                                                object
Cause of Death                                                           object
Cause of Death Group \n(Used in the NPS Mortality Dashboard)             object
Intent                                                                   object
Outcome                                                                  object
Sex                                                                      object
Age Range                                                                object
Activity                                                                 object
original_name                                                            object
Age Range Min                                                           float64
Age Range Max                                                           float64
dtype: object

In [162]:
print(df[["Age Range","Age Range Min","Age Range Max"]].drop_duplicates())

   Age Range  Age Range Min  Age Range Max
0        65+           65.0            NaN
1    Unknown            NaN            NaN
3      15-24           15.0           24.0
4      45-54           45.0           54.0
7      25-34           25.0           34.0
10     55-64           55.0           64.0
11     35-44           35.0           44.0
13      0-14            0.0           14.0


Now I'll look at the outcome column.

In [163]:
df["Outcome"].unique()

array(['fatal injury'], dtype=object)

This column doesn't add anything. We know each event was a fatal injury. I'm going to drop the entire column.

In [164]:
df.drop("Outcome", axis=1, inplace=True)

In [165]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Sex,Age Range,Activity,original_name,Age Range Min,Age Range Max
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,Glen Canyon National Recreation Area,65.0,NaN
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,male,Unknown,vessel related,Golden Gate National Recreation Area,NaN,NaN
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,male,Unknown,vessel related,Golden Gate National Recreation Area,NaN,NaN
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,15-24,driving,Natchez Trace Parkway,15.0,24.0
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,45-54,driving,Natchez Trace Parkway,45.0,54.0


Now I'll look at "Intent"

In [166]:
df["Intent"].value_counts()

Intent
unintentional    2713
intentional       701
medical           640
undetermined      581
Name: count, dtype: int64

I want to see if the "Medical" rows all correspond with the various medical causes of death. I'm going to normalize the Intent and Cause of Death columns and then compare to how many I would expect with the medical categories (3).

In [167]:
#See what causes appear under Intent == "Medical"
medical_causes_found = (
    df.loc[df["Intent"] == "medical", "Cause of Death"]
      .dropna()
      .unique()
)
print(medical_causes_found)

['medical - not during physical activity'
 'medical - during physical activity' 'medical - unknown']


This is what I expected. I'm just going to double check that the number of medical intent fields matches the sum of the three types of medical cause fields.

In [168]:
medical_causes = [
    "medical - not during physical activity",
    "medical - during physical activity",
    "medical - unknown"
]

df["Cause of Death"].value_counts().loc[medical_causes]

Cause of Death
medical - not during physical activity    240
medical - during physical activity        318
medical - unknown                          82
Name: count, dtype: int64

There are 640 medical causes, and 640 medical intents, so I believe they corrospond correctly. 

In [169]:
df["Sex"].value_counts()

Sex
male            3484
female           856
not reported     295
Name: count, dtype: int64

These are the values we would expect and there are no missing values. No additional work is necessary.

In [170]:
df["Activity"].sort_values().unique()

array(['base jumping', 'bicycling', 'camping', 'canyoneering', 'caving',
       'climbing', 'crossing river', 'diving (land)', 'diving - free',
       'diving - scuba', 'driving', 'eating', 'fishing', 'flying',
       'hiking', 'homicide', 'horseback (or mule) riding', 'hunting',
       'illegal activity', 'not reported', 'other', 'photographing',
       'playing sports', 'rock scrambeling', 'rock scrambling', 'sitting',
       'skateboarding', 'skiing', 'sleeping', 'snorkeling',
       'snowboarding', 'snowmobiling', 'snowshoeing', 'suicide',
       'swimming', 'undetermined', 'vessel related', 'wading', 'walking',
       'wildlife watching'], dtype=object)

I can combine "Rock Scrambeling" and "Rock Scrambling". I considered combining "Not Reported" and "Undetermined, but chose not to at this time because they do convey different information.

In [171]:
df["Activity"] = df["Activity"].replace(
    "Rock Scrambeling", "Rock Scrambling"
)

df["Activity"].sort_values().unique()

array(['base jumping', 'bicycling', 'camping', 'canyoneering', 'caving',
       'climbing', 'crossing river', 'diving (land)', 'diving - free',
       'diving - scuba', 'driving', 'eating', 'fishing', 'flying',
       'hiking', 'homicide', 'horseback (or mule) riding', 'hunting',
       'illegal activity', 'not reported', 'other', 'photographing',
       'playing sports', 'rock scrambeling', 'rock scrambling', 'sitting',
       'skateboarding', 'skiing', 'sleeping', 'snorkeling',
       'snowboarding', 'snowmobiling', 'snowshoeing', 'suicide',
       'swimming', 'undetermined', 'vessel related', 'wading', 'walking',
       'wildlife watching'], dtype=object)

In [172]:
df.head()

,Incident Date,Park Name,Cause of Death,Cause of Death Group \n(Used in the NPS Mortality Dashboard),Intent,Sex,Age Range,Activity,original_name,Age Range Min,Age Range Max
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,Glen Canyon National Recreation Area,65.0,NaN
1,2007-01-22,golden gate national recreation area,drowning,drowning,unintentional,male,Unknown,vessel related,Golden Gate National Recreation Area,NaN,NaN
2,2007-01-22,golden gate national recreation area,undetermined,undetermined,undetermined,male,Unknown,vessel related,Golden Gate National Recreation Area,NaN,NaN
3,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,15-24,driving,Natchez Trace Parkway,15.0,24.0
4,2007-01-29,natchez trace parkway,motor vehicle crash,motor vehicle crash,unintentional,female,45-54,driving,Natchez Trace Parkway,45.0,54.0


In [173]:
df.columns

Index(['Incident Date', 'Park Name', 'Cause of Death',
       'Cause of Death Group \n(Used in the NPS Mortality Dashboard) ',
       'Intent', 'Sex', 'Age Range', 'Activity', 'original_name',
       'Age Range Min', 'Age Range Max'],
      dtype='object')

In [174]:
df = df.rename(columns={
    "Incident Date": "incident_date",
    "Park Name": "mortality_name", 
    "Cause of Death Group \n(Used in the NPS Mortality Dashboard) ": "cause_group",
    "Intent": "intent",
    "Sex": "sex",
    "Age Range": "age_range",
    "Activity": "activity",
    "Age Range Min": "age_range_min",
    "Age Range Max": "age_range_max",
    "Cause of Death": "cause"
    })

df.columns

Index(['incident_date', 'mortality_name', 'cause', 'cause_group', 'intent',
       'sex', 'age_range', 'activity', 'original_name', 'age_range_min',
       'age_range_max'],
      dtype='object')

I know that NPS visitation data is available in monthly increments only, so I'm going to pull the month and year out of the datetime field to create new columns I can use when I compare visitation data.

In [175]:
df["month"] = df["incident_date"].dt.month
df["year"]  = df["incident_date"].dt.year
df.head(1)

,incident_date,mortality_name,cause,cause_group,intent,sex,age_range,activity,original_name,age_range_min,age_range_max,month,year
0,2007-01-01,glen canyon national recreation area,undetermined,undetermined,undetermined,male,65+,not reported,Glen Canyon National Recreation Area,65.0,NaN,1,2007


I'm going to prepare the tables I'll use in my dataframe.

In [176]:
#Unique causes of death and categories to make querying by cause easier
unique_causes = df['cause'].dropna().unique()
df_cause_types = pd.DataFrame({
    'cause_id': range(1, len(unique_causes) + 1),
    'cause_name': sorted(unique_causes)
})

# Get the cause_group for each cause
cause_group_map = df.groupby('cause')['cause_group'].first()
df_cause_types['cause_category'] = df_cause_types['cause_name'].map(cause_group_map)

df_cause_types.head()

,cause_id,cause_name,cause_category
0,1,aircraft incident,other transportation
1,2,altitude,environmental
2,3,asphyxiation,other
3,4,avalanche,environmental
4,5,bicycle only crash,other transportation


In [177]:
#unique intents
unique_intents = df['intent'].dropna().unique()
df_intent_types = pd.DataFrame({
    'intent_id': range(1, len(unique_intents) + 1),
    'intent_name': sorted(unique_intents)
})

df_intent_types.head()

,intent_id,intent_name
0,1,intentional
1,2,medical
2,3,undetermined
3,4,unintentional


In [178]:
unique_sex = df['sex'].dropna().unique()
df_sex_types = pd.DataFrame({
    'sex_id': range(1, len(unique_sex) + 1),
    'sex_name': sorted(unique_sex)
})

df_sex_types.head()

,sex_id,sex_name
0,1,female
1,2,male
2,3,not reported


In [179]:
unique_activities = df['activity'].dropna().unique()
df_activity_types = pd.DataFrame({
    'activity_id': range(1, len(unique_activities) + 1),
    'activity_name': sorted(unique_activities)
})

df_activity_types.head()

,activity_id,activity_name
0,1,base jumping
1,2,bicycling
2,3,camping
3,4,canyoneering
4,5,caving


In [180]:
unique_parks = df['mortality_name'].dropna().unique()
park_names = df.groupby('mortality_name')['original_name'].first()

df_parks = pd.DataFrame({
    'park_id': range(1, len(unique_parks) + 1),
    'park_code': sorted(unique_parks),
    'official_name': [park_names[name] for name in sorted(unique_parks)]
})

df_parks.head(1)

,park_id,park_code,official_name
0,1,acadia national park,Acadia National Park


In [181]:
# Get date range
start_date = pd.to_datetime(df['incident_date']).min()
end_date = pd.to_datetime(df['incident_date']).max()

# Create date range
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Create date dimension dataframe
df_dates = pd.DataFrame({
    'date_id': range(1, len(date_range) + 1),
    'date': date_range,
    'year': date_range.year,
    'month': date_range.month,
    'day': date_range.day,
    'day_of_week': date_range.dayofweek,  # 0=Monday, 6=Sunday
    'day_name': date_range.day_name(),
    'quarter': date_range.quarter,
    'is_weekend': date_range.dayofweek.isin([5, 6]).astype(int),
})

# Add season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_dates['season'] = df_dates['month'].apply(get_season)

df_dates.head()

,date_id,date,year,month,day,day_of_week,day_name,quarter,is_weekend,season
0,1,2007-01-01,2007,1,1,0,Monday,1,0,Winter
1,2,2007-01-02,2007,1,2,1,Tuesday,1,0,Winter
2,3,2007-01-03,2007,1,3,2,Wednesday,1,0,Winter
3,4,2007-01-04,2007,1,4,3,Thursday,1,0,Winter
4,5,2007-01-05,2007,1,5,4,Friday,1,0,Winter


In [182]:
df_dates.tail()

,date_id,date,year,month,day,day_of_week,day_name,quarter,is_weekend,season
6385,6386,2024-06-25,2024,6,25,1,Tuesday,2,0,Summer
6386,6387,2024-06-26,2024,6,26,2,Wednesday,2,0,Summer
6387,6388,2024-06-27,2024,6,27,3,Thursday,2,0,Summer
6388,6389,2024-06-28,2024,6,28,4,Friday,2,0,Summer
6389,6390,2024-06-29,2024,6,29,5,Saturday,2,1,Summer


In [183]:
df_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6390 entries, 0 to 6389
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date_id      6390 non-null   int64         
 1   date         6390 non-null   datetime64[ns]
 2   year         6390 non-null   int32         
 3   month        6390 non-null   int32         
 4   day          6390 non-null   int32         
 5   day_of_week  6390 non-null   int32         
 6   day_name     6390 non-null   object        
 7   quarter      6390 non-null   int32         
 8   is_weekend   6390 non-null   int64         
 9   season       6390 non-null   object        
dtypes: datetime64[ns](1), int32(5), int64(2), object(2)
memory usage: 374.5+ KB


In [184]:
#create mapping tables
cause_to_id = dict(zip(df_cause_types['cause_name'], df_cause_types['cause_id']))
intent_to_id = dict(zip(df_intent_types['intent_name'], df_intent_types['intent_id']))
sex_to_id = dict(zip(df_sex_types['sex_name'], df_sex_types['sex_id']))
activity_to_id = dict(zip(df_activity_types['activity_name'], df_activity_types['activity_id']))
park_to_id = dict(zip(df_parks['park_code'], df_parks['park_id']))
date_to_id = dict(zip(df_dates['date'].dt.date, df_dates['date_id']))

In [185]:
#create df_mortality_incidents that uses keys from lookup tables
df_mortality_incidents = pd.DataFrame({
    'incident_id': range(1, len(df) + 1),
    'park_id': df['mortality_name'].map(park_to_id),
    'mortality_name': df['mortality_name'],
    'original_name': df['original_name'],
    'incident_date': df['incident_date'],
    'date_id': df['incident_date'].map(date_to_id),
    'cause_id': df['cause'].map(cause_to_id),
    'intent_id': df['intent'].map(intent_to_id),
    'sex_id': df['sex'].map(sex_to_id),
    'activity_id': df['activity'].map(activity_to_id),
    'age_range_original': df['age_range'],
    'age_min': df['age_range_min'],
    'age_max': df['age_range_max'],
})

df_mortality_incidents.head()

,incident_id,park_id,mortality_name,original_name,incident_date,date_id,cause_id,intent_id,sex_id,activity_id,age_range_original,age_min,age_max
0,1,82,glen canyon national recreation area,Glen Canyon National Recreation Area,2007-01-01,1,39,3,2,20,65+,65.0,NaN
1,2,84,golden gate national recreation area,Golden Gate National Recreation Area,2007-01-22,22,7,4,2,37,Unknown,NaN,NaN
2,3,84,golden gate national recreation area,Golden Gate National Recreation Area,2007-01-22,22,39,3,2,37,Unknown,NaN,NaN
3,4,139,natchez trace parkway,Natchez Trace Parkway,2007-01-29,29,24,4,1,11,15-24,15.0,24.0
4,5,139,natchez trace parkway,Natchez Trace Parkway,2007-01-29,29,24,4,1,11,45-54,45.0,54.0


Unfortunately, there are several unique visit park codes assigned to some parks. The park code is a key required to interact with may NPS APIs. There is not a complete listing of park codes available but I know I want to obtain the full visitation data set, so I've found a drop down list with all the codes used by this dataset. I used the inspect panel in a Windows browser to copy the entries from the bounded list, which is saved in my data folder as visit_codes.html. You can obtain updated information using the same process at https://irma.nps.gov/Stats/Reports/Park

In [186]:
with open("../data/visit_codes.html", "r", encoding="utf-8") as f:
    html = f.read()

print(len(html))
print(html[:500])

88925
<li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" data-recordindex="0" data-recordid="1" data-boundview="filtercombobox-1012-picker" aria-selected="false">Abraham Lincoln Birthplace NHP (ABLI)</li><li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" data-recordindex="1" data-recordid="2" data-boundview="filtercombobox-1012-picker" aria-selected="false">Acadia NP (ACAD)</li><li role="option" unselectable="on" class="x-boundlist-item" tabindex="-1" da


Now I'm going to use BeautifulSoup to find all the items in the boundlist that I copied. This should be all my parks. I'm going to look at the length to see if it's working correctly. Each item that has a class="x-boundlist-item" should be an individual park, so I'm going to divide those into a list and see how many there are. I'm expcting just a little over 400, because that's how many national park units I know exist.

In [187]:
soup = BeautifulSoup(html, "html.parser")

items = soup.find_all("li", class_="x-boundlist-item")
print(len(items))
print(items[0:5])

415
[<li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="1" data-recordindex="0" role="option" tabindex="-1" unselectable="on">Abraham Lincoln Birthplace NHP (ABLI)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="2" data-recordindex="1" role="option" tabindex="-1" unselectable="on">Acadia NP (ACAD)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="3" data-recordindex="2" role="option" tabindex="-1" unselectable="on">Adams NHP (ADAM)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="4" data-recordindex="3" role="option" tabindex="-1" unselectable="on">African Burial Ground NM (AFBG)</li>, <li aria-selected="false" class="x-boundlist-item" data-boundview="filtercombobox-1012-picker" data-recordid="5" data-recordindex="4" role="option"

Now that it's divided into each list item instead of a giant string of code, I'm going to put it into rows.

In [188]:
#create blank place to hold the rows when they're created
rows = []

#create for loop that will go through all of the boundlist items 
for li in items:
    #get just the text that would display withough the rest of the code
    text = li.get_text(strip=True)   

    #I'm going to use a regular expression to seperate the part in parentheses from the rest
    m = re.match(r"^(.*)\(([^)]+)\)$", text)
    if m:
        name = m.group(1).strip()
        code = m.group(2).strip()
        rows.append({"UnitName": name, "UnitCode": code})
    else:
        # in case some weird row doesn't match that pattern
        rows.append({"UnitName": text, "UnitCode": None})


In [189]:
df_visit_codes = pd.DataFrame(rows)

df_visit_codes.head()

,UnitName,UnitCode
0,Abraham Lincoln Birthplace NHP,ABLI
1,Acadia NP,ACAD
2,Adams NHP,ADAM
3,African Burial Ground NM,AFBG
4,Agate Fossil Beds NM,AGFO


In [190]:
df_visit_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   UnitName  415 non-null    object
 1   UnitCode  415 non-null    object
dtypes: object(2)
memory usage: 6.6+ KB


In [191]:
df_visit_codes = df_visit_codes.rename(
    columns={"UnitCode": "visits_code",
    "UnitName": "visits_name"})

In [192]:
df_visit_codes["Name"]=df_visit_codes["visits_name"].copy()

df_visit_codes.head(1)

,visits_name,visits_code,Name
0,Abraham Lincoln Birthplace NHP,ABLI,Abraham Lincoln Birthplace NHP


In [193]:
cols = ["visits_name", 
        "visits_code"]

df_visit_codes[cols] = df_visit_codes[cols].apply(clean)

In [194]:
df_visit_codes.head(1)

,visits_name,visits_code,Name
0,abraham lincoln birthplace nhp,abli,Abraham Lincoln Birthplace NHP


I'll use the API to call for all the parks available in the list and obtain all available visitor data, which I will then map to my dataset.

In [195]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"

all_codes = (
    df_visit_codes["visits_code"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
    .tolist()
)

def chunk_list(lst, n):
    #Yield successive n-sized chunks from lst.
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

CODES_PER_CALL = 50

all_results = []

with requests.Session() as s:
    for chunk in chunk_list(all_codes, CODES_PER_CALL):
        params = {
            "unitCodes": ",".join(chunk),
            "startMonth": 1,
            "startYear": 2007,
            "endMonth": 6,
            "endYear": 2024,
            "format": "json",
        }
        r = s.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)
        r.raise_for_status()
        data = r.json()
        all_results.extend(data)

df_visits = pd.DataFrame(all_results)

In [196]:
df_visits.head()

,Month,NonRecreationVisitors,RecreationVisitors,UnitCode,UnitName,Year
0,1,0,4960,ABLI,Abraham Lincoln Birthplace NHP,2007
1,2,0,5877,ABLI,Abraham Lincoln Birthplace NHP,2007
2,3,0,9868,ABLI,Abraham Lincoln Birthplace NHP,2007
3,4,0,17900,ABLI,Abraham Lincoln Birthplace NHP,2007
4,5,0,21277,ABLI,Abraham Lincoln Birthplace NHP,2007


In [197]:
df_visits["Name"] = df_visits["UnitName"].copy()

In [198]:
cols = ["UnitCode", 
        "UnitName", 
        ]

df_visits[cols] = df_visits[cols].apply(clean)

df_visits.head(1)

,Month,NonRecreationVisitors,RecreationVisitors,UnitCode,UnitName,Year,Name
0,1,0,4960,abli,abraham lincoln birthplace nhp,2007,Abraham Lincoln Birthplace NHP


In [199]:
df_visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78652 entries, 0 to 78651
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Month                  78652 non-null  int64 
 1   NonRecreationVisitors  78652 non-null  int64 
 2   RecreationVisitors     78652 non-null  int64 
 3   UnitCode               78652 non-null  object
 4   UnitName               78652 non-null  object
 5   Year                   78652 non-null  int64 
 6   Name                   78652 non-null  object
dtypes: int64(4), object(3)
memory usage: 4.2+ MB


In [200]:
df_visits = df_visits.rename(
    columns = {"RecreationVisitors": "recreation_visitors",
               "NonRecreationVisitors": "nonrecreation_visitors",
               "UnitCode": "visits_code",
               "UnitName": "visits_name",
               "Month": "month",
               "Year": "year",
               "Name": "name"}
)

df_visits.head(1)

,month,nonrecreation_visitors,recreation_visitors,visits_code,visits_name,year,name
0,1,0,4960,abli,abraham lincoln birthplace nhp,2007,Abraham Lincoln Birthplace NHP


In [201]:
df_visits.to_csv("../data/NPS_visitation_data.csv", index=False)

In [202]:
df_visits["visits_code"].value_counts().head()

visits_code
zion    210
abli    210
acad    210
adam    210
yuch    210
Name: count, dtype: int64

In [203]:
df_visits["visits_code"].value_counts().tail()

visits_code
frst    18
stge     6
till     6
iatr     6
amch     6
Name: count, dtype: int64

In [204]:
counts = df_visits["visits_code"].value_counts()

len(counts[counts != 210])

38

Since we are missing some parks entirely and some don't have complete data, I'm going to do a couple of sanity checks. First, I'll call the API for one of the unit codes that didn't have any data, and then for one with incomplete data. I want to ensure the data is actually missing and there wasn't a problem with my API call.

In [205]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"
params = {
    "unitCodes": "AMCH",
    "startMonth": 1, "startYear": 2007,
    "endMonth": 6,   "endYear": 2024,
    "format": "json",
}

r = requests.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)

In [206]:
AMCH = r.json()
len(AMCH)

6

In [207]:
missing_visits = set(df_visit_codes["visits_code"]) - set(df_visits["visits_code"])
print(missing_visits)

{'fomr', 'spra', 'hono', 'camo', 'blrv', 'jofk', 'brcr', 'hart', 'naca', 'ebla', 'cebe', 'neph', 'frri', 'okci', 'tupe', 'bicr', 'navc'}


In [208]:
BASE = "https://irmaservices.nps.gov/v3/rest/stats/visitation"
params = {
    "unitCodes": "NEPH",
    "startMonth": 1, "startYear": 2007,
    "endMonth": 6,   "endYear": 2024,
    "format": "json",
}

r = requests.get(BASE, params=params, headers={"Accept": "application/json"}, timeout=30)

NEPH = r.json()
len(NEPH)

0

The data appears to be truly missing.

To obtain the rest of the data, I need to compile a master list of all park codes that could be found. I'll use this both to build the API query and to merge obtained datasets. I'll start with the current, official unit list. The list is saved in the data folder of this project as NPS-Unit-List.xlxs, and can be obtained by downloading from this page: https://www.nps.gov/aboutus/foia/foia-reading-room.htm

In [209]:
df_units = pd.read_excel("../data/NPS-Unit-List.xlsx")

df_units.head()

,Name,Type,Park Code,Org Code,Region
0,NPS unit type designations (link),NaN,NaN,NaN,NaN
1,ABRAHAM LINCOLN BIRTHPLACE,NHP,ABLI,5540,SER
2,ACADIA,NP,ACAD,1700,NER
3,ADAMS,NHP,ADAM,1710,NER
4,AFRICAN BURIAL GROUND,NM,AFBG,1762,NER


In [210]:
df_units.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 428 entries, 0 to 427
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Name       428 non-null    object
 1   Type       424 non-null    object
 2   Park Code  425 non-null    object
 3   Org Code   368 non-null    object
 4   Region     425 non-null    object
dtypes: object(5)
memory usage: 16.8+ KB


In [211]:
df_units[df_units["Park Code"].isna()]["Name"]

0      NPS unit type designations (link)
262              MARTIN LUTHER KING, JR.
417                          WORLD WAR I
Name: Name, dtype: object

In [212]:
df_units = df_units.drop(df_units.index[0])

df_units.head()

,Name,Type,Park Code,Org Code,Region
1,ABRAHAM LINCOLN BIRTHPLACE,NHP,ABLI,5540,SER
2,ACADIA,NP,ACAD,1700,NER
3,ADAMS,NHP,ADAM,1710,NER
4,AFRICAN BURIAL GROUND,NM,AFBG,1762,NER
5,AGATE FOSSIL BEDS,NM,AGFO,6710,MWR


In [213]:
cols = ["Name", 
        "Type", 
        "Park Code", 
        "Org Code", 
        "Region"]

df_units[cols] = df_units[cols].apply(clean)

df_units.head()

,Name,Type,Park Code,Org Code,Region
1,abraham lincoln birthplace,nhp,abli,5540,ser
2,acadia,np,acad,1700,ner
3,adams,nhp,adam,1710,ner
4,african burial ground,nm,afbg,1762,ner
5,agate fossil beds,nm,agfo,6710,mwr


In [214]:
df_units = df_units.rename(columns={
    "Name": "units_name",
    "Park Code": "units_code",
    "Type": "type",
    "Org Code": "org_code",
    "Region": "region"
})

df_units.head(1)

,units_name,type,units_code,org_code,region
1,abraham lincoln birthplace,nhp,abli,5540,ser


Because park codes have changed overtime and records may not necessarily be updated, I will also use a historic listing of codes to ensure I am able to obtain all the data I need. I will scrape this information from the table shown at https://www.nps.gov/articles/000/historic-listing-of-nps-park-codes.htm

In [215]:
url = "https://www.nps.gov/articles/000/historic-listing-of-nps-park-codes.htm"
tables = pd.read_html(url)
print(len(tables))          # how many tables pandas found

1


In [216]:
df_historic_codes = tables[0]
df_historic_codes.head()

,0,1
0,"Park, Program, or Office Name",Alpha Code
1,Abraham Lincoln Birthplace National Historical...,ABLI
2,Abraham Lincoln National Heritage Area,See MWR
3,Acadia National Park,ACAD
4,"Adams Memorial, District of Columbia",See NCR


In [217]:
df_historic_codes.columns = df_historic_codes.iloc[0]
df_historic_codes = df_historic_codes.drop(df_historic_codes.index[0])
df_historic_codes = df_historic_codes.reset_index(drop=True)
df_historic_codes.columns = df_historic_codes.columns.str.strip()
df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code
0,Abraham Lincoln Birthplace National Historical...,ABLI
1,Abraham Lincoln National Heritage Area,See MWR
2,Acadia National Park,ACAD
3,"Adams Memorial, District of Columbia",See NCR
4,Adams National Historical Park,ADAM


In [218]:
df_historic_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Park, Program, or Office Name  614 non-null    object
 1   Alpha Code                     614 non-null    object
dtypes: object(2)
memory usage: 9.7+ KB


I need to clean up the Alpha Code column based on what I can see on the site. First, I'll remove "See " from the beginning of rows that are alternate or historic names. I'll start by creating a new column called "Alternate Name" with a boolean value that designates those rows, and then drop the text "See" from the "Alpha Code" column.

In [219]:
df_historic_codes["Alternate Name"] = (
    df_historic_codes["Alpha Code"].str.startswith("See ")
)

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name
0,Abraham Lincoln Birthplace National Historical...,ABLI,False
1,Abraham Lincoln National Heritage Area,See MWR,True
2,Acadia National Park,ACAD,False
3,"Adams Memorial, District of Columbia",See NCR,True
4,Adams National Historical Park,ADAM,False


In [220]:
#remove "See" from Alpha column
df_historic_codes["Alpha Code"] = (
    df_historic_codes["Alpha Code"].str.replace(r"^See\s+", "", regex=True)
)

# Create a Region column for the 3-letter codes
df_historic_codes["Region"] = df_historic_codes["Alpha Code"].where(
    df_historic_codes["Alpha Code"].str.fullmatch(r"[A-Za-z]{3}")
)

# Drop 3-letter codes from Alpha Code (set to NaN)
df_historic_codes.loc[
    df_historic_codes["Alpha Code"].str.fullmatch(r"[A-Za-z]{3}"),
    "Alpha Code"
] = pd.NA

df_historic_codes.head()


,"Park, Program, or Office Name",Alpha Code,Alternate Name,Region
0,Abraham Lincoln Birthplace National Historical...,ABLI,False,NaN
1,Abraham Lincoln National Heritage Area,<NA>,True,MWR
2,Acadia National Park,ACAD,False,NaN
3,"Adams Memorial, District of Columbia",<NA>,True,NCR
4,Adams National Historical Park,ADAM,False,NaN


In [221]:
# Convert both columns to object dtype
df_historic_codes["Alpha Code"] = df_historic_codes["Alpha Code"].astype(object)
df_historic_codes["Region"] = df_historic_codes["Region"].astype(object)

df_historic_codes = df_historic_codes.replace({pd.NA: np.nan})

df_historic_codes.head()


,"Park, Program, or Office Name",Alpha Code,Alternate Name,Region
0,Abraham Lincoln Birthplace National Historical...,ABLI,False,NaN
1,Abraham Lincoln National Heritage Area,NaN,True,MWR
2,Acadia National Park,ACAD,False,NaN
3,"Adams Memorial, District of Columbia",NaN,True,NCR
4,Adams National Historical Park,ADAM,False,NaN


In [222]:
df_historic_codes["historic_name"] = df_historic_codes["Park, Program, or Office Name"]

In [223]:
cols = ["historic_name", 
        "Alpha Code", 
        "Region"]

df_historic_codes[cols] = df_historic_codes[cols].apply(clean)

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name,Region,historic_name
0,Abraham Lincoln Birthplace National Historical...,abli,False,<NA>,abraham lincoln birthplace national historical...
1,Abraham Lincoln National Heritage Area,<NA>,True,mwr,abraham lincoln national heritage area
2,Acadia National Park,acad,False,<NA>,acadia national park
3,"Adams Memorial, District of Columbia",<NA>,True,ncr,"adams memorial, district of columbia"
4,Adams National Historical Park,adam,False,<NA>,adams national historical park


In [224]:
df_historic_codes[
    df_historic_codes["Alpha Code"].notna() &
    (df_historic_codes["Alpha Code"].str.len() != 4)
]["Alpha Code"].unique()

array(['cave (formerly caca)', 'use mwr', 'drto (formerly foje)',
       'use stli', 'use timu', 'colo, jame', 'jnem, jeff',
       'lake (formerly lame)', 'grte or yell', 'colo, york'], dtype=object)

In [225]:
#Drop rows with "use"
mask_use = df_historic_codes["Alpha Code"].str.contains(r"\buse\b", case=False, na=False)
df_historic_codes = df_historic_codes[~mask_use].copy()

#Handle "formerly" rows
former = df_historic_codes["Alpha Code"].str.extract(
    r"^\s*(?P<current>[A-Za-z0-9]{4})\s*\(formerly\s+(?P<former>[A-Za-z0-9]{4})\)\s*$",
    expand=True
)

#Add "Former Code" column
df_historic_codes["Former Code"] = former["former"]

#Replace "Alpha Code" with current code
df_historic_codes.loc[former["current"].notna(), "Alpha Code"] = former["current"]

#Handle "or" rows
alts = df_historic_codes["Alpha Code"].str.extract(
    r"^\s*(?P<primary>[A-Za-z0-9]{4})\s*(?:,|or)\s*(?P<alt>[A-Za-z0-9]{4})\s*$",
    expand=True
)

#Add Alternate Code column
df_historic_codes["Alternate Code"] = alts["alt"]

#Replace "Alpha Code" with first code
df_historic_codes.loc[alts["primary"].notna(), "Alpha Code"] = alts["primary"]

df_historic_codes.head()

,"Park, Program, or Office Name",Alpha Code,Alternate Name,Region,historic_name,Former Code,Alternate Code
0,Abraham Lincoln Birthplace National Historical...,abli,False,<NA>,abraham lincoln birthplace national historical...,NaN,NaN
1,Abraham Lincoln National Heritage Area,<NA>,True,mwr,abraham lincoln national heritage area,NaN,NaN
2,Acadia National Park,acad,False,<NA>,acadia national park,NaN,NaN
3,"Adams Memorial, District of Columbia",<NA>,True,ncr,"adams memorial, district of columbia",NaN,NaN
4,Adams National Historical Park,adam,False,<NA>,adams national historical park,NaN,NaN


In [226]:
df_historic_codes[
    df_historic_codes["Alpha Code"].notna() &
    (df_historic_codes["Alpha Code"].str.len() != 4)
]["Alpha Code"].unique()

array([], dtype=object)

In [227]:
df_historic_codes = df_historic_codes.rename(columns={
    "Park, Program, or Office Name": "name",
    "Alpha Code": "historic_code",
    "Alternate Name": "alternate_name",
    "Former Code": "former_code",
    "Alternate Code": "alternate_code",
    "Region": "region"
    
})

df_historic_codes.head(1)

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
0,Abraham Lincoln Birthplace National Historical...,abli,False,<NA>,abraham lincoln birthplace national historical...,NaN,NaN


There are some duplicate codes in rows because the historic listing has prior names/info that has changed, while the code did not change. I want to look more closely at those so I can merge cleanly.

In [228]:
#Look at codes that have multiple rows
duplicate_codes = df_historic_codes['historic_code'].value_counts()
duplicate_codes = duplicate_codes[duplicate_codes > 1].index

duplicate_codes

Index(['gaar', 'colo', 'grte', 'seki', 'noca', 'lacl', 'gwmp', 'grsa', 'grba',
       'caha', 'glac', 'iatr', 'ania', 'cuva', 'knri', 'kova', 'yose', 'nace',
       'nama', 'orca', 'valr', 'qush'],
      dtype='object', name='historic_code')

I'm going to cycle through these to double check that the false value in the alternateName column is in the correct row. For brevity, I've only left the parks that needed adjusments after checking each manually.

In [229]:
df_historic_codes[df_historic_codes["historic_code"] == "caha"]

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
79,Cape Hatteras National Seashore,caha,False,<NA>,cape hatteras national seashore,NaN,NaN
80,"Cape Henry Memorial, Part of Colonial National...",caha,False,<NA>,"cape henry memorial, part of colonial national...",NaN,NaN


In [230]:
df_historic_codes.at[80, 'alternate_name'] = True

df_historic_codes[df_historic_codes["historic_code"] == "caha"]

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
79,Cape Hatteras National Seashore,caha,False,<NA>,cape hatteras national seashore,NaN,NaN
80,"Cape Henry Memorial, Part of Colonial National...",caha,True,<NA>,"cape henry memorial, part of colonial national...",NaN,NaN


In [231]:
df_historic_codes[df_historic_codes["historic_code"] == "colo"]

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
118,Colonial National Historical Park,colo,False,<NA>,colonial national historical park,NaN,NaN
278,Jamestown National Historic Site (part of Colo...,colo,False,<NA>,jamestown national historic site (part of colo...,NaN,jame
608,Yorktown Battlefield (part of Colonial Nationa...,colo,False,<NA>,yorktown battlefield (part of colonial nationa...,NaN,york


In [232]:
df_historic_codes.loc[[278, 608], 'alternate_name'] = True

df_historic_codes[df_historic_codes["historic_code"] == "colo"]

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
118,Colonial National Historical Park,colo,False,<NA>,colonial national historical park,NaN,NaN
278,Jamestown National Historic Site (part of Colo...,colo,True,<NA>,jamestown national historic site (part of colo...,NaN,jame
608,Yorktown Battlefield (part of Colonial Nationa...,colo,True,<NA>,yorktown battlefield (part of colonial nationa...,NaN,york


Next, I'll obtain the full list of park units from the parks API. This API does require a key, which can obtained by registering through this link: https://www.nps.gov/subjects/developer/guides.htm

In [233]:
def fetch_all_parks(api_key: str, page_size: int = 50, pause: float = 0.2) -> pd.DataFrame:
    base = "https://developer.nps.gov/api/v1/parks"
    start = 0
    rows = []

#Create a loop that will continue until there is no data or the data returned is < the page size
    while True:
        params = {"limit": page_size, "start": start, "api_key": api_key}
        resp = requests.get(base, params=params, timeout=30)
        resp.raise_for_status()
        #convert the response from JSON text to Python dictionary
        payload = resp.json()

        data = payload.get("data", [])
        if not data:
            break
        
        #The default parks response has more info than I need, so I'm going to filter just for the name and park code
        for item in data:
            rows.append({
                "api_name": item.get("fullName"),
                "api_code": item.get("parkCode"),
                "description": item.get("description"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "weather": item.get("weatherInfo"),
                "name": item.get(name),
                "designation": item.get("designation")
            })

        if len(data) < page_size:
            break

        start += page_size
        time.sleep(pause)

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

df_api_codes = fetch_all_parks(API_KEY, page_size=50)

df_api_codes.head()

,api_name,api_code,description,latitude,longitude,weather,name,designation
0,Abraham Lincoln Birthplace National Historical...,abli,For over a century people from around the worl...,37.5858662,-85.67330523,There are four distinct seasons in Central Ken...,None,National Historical Park
1,Acadia National Park,acad,Acadia National Park protects the natural beau...,44.409286,-68.247501,"Located on Mount Desert Island in Maine, Acadi...",None,National Park
2,Adams National Historical Park,adam,From the sweet little farm at the foot of Penn...,42.2553961,-71.01160356,"Be prepared for hot, humid weather during the ...",None,National Historical Park
3,African American Civil War Memorial,afam,"Over 200,000 African-American soldiers and sai...",38.9166,-77.026,Washington DC gets to see all four seasons. Hu...,None,
4,African Burial Ground National Monument,afbg,The African Burial Ground is the oldest and la...,40.71452681,-74.00447358,http://forecast.weather.gov/MapClick.php?CityN...,None,National Monument


In [234]:
df_api_codes["original_api_name"] = df_api_codes["api_name"]

df_api_codes.head(1)

,api_name,api_code,description,latitude,longitude,weather,name,designation,original_api_name
0,Abraham Lincoln Birthplace National Historical...,abli,For over a century people from around the worl...,37.5858662,-85.67330523,There are four distinct seasons in Central Ken...,None,National Historical Park,Abraham Lincoln Birthplace National Historical...


In [235]:
cols = ["api_name", 
        "api_code",
        "name",
        "designation"]

df_api_codes[cols] = df_api_codes[cols].apply(clean)

df_api_codes.head()

,api_name,api_code,description,latitude,longitude,weather,name,designation,original_api_name
0,abraham lincoln birthplace national historical...,abli,For over a century people from around the worl...,37.5858662,-85.67330523,There are four distinct seasons in Central Ken...,<NA>,national historical park,Abraham Lincoln Birthplace National Historical...
1,acadia national park,acad,Acadia National Park protects the natural beau...,44.409286,-68.247501,"Located on Mount Desert Island in Maine, Acadi...",<NA>,national park,Acadia National Park
2,adams national historical park,adam,From the sweet little farm at the foot of Penn...,42.2553961,-71.01160356,"Be prepared for hot, humid weather during the ...",<NA>,national historical park,Adams National Historical Park
3,african american civil war memorial,afam,"Over 200,000 African-American soldiers and sai...",38.9166,-77.026,Washington DC gets to see all four seasons. Hu...,<NA>,<NA>,African American Civil War Memorial
4,african burial ground national monument,afbg,The African Burial Ground is the oldest and la...,40.71452681,-74.00447358,http://forecast.weather.gov/MapClick.php?CityN...,<NA>,national monument,African Burial Ground National Monument


Now that I've obtained codes from every source I can find, I am going to write a function that creates a list of all available codes. I'll use this to query the remaining APIs.

In [236]:
df_visits.columns

Index(['month', 'nonrecreation_visitors', 'recreation_visitors', 'visits_code',
       'visits_name', 'year', 'name'],
      dtype='object')

In [237]:
df_api_codes.head(1)

,api_name,api_code,description,latitude,longitude,weather,name,designation,original_api_name
0,abraham lincoln birthplace national historical...,abli,For over a century people from around the worl...,37.5858662,-85.67330523,There are four distinct seasons in Central Ken...,<NA>,national historical park,Abraham Lincoln Birthplace National Historical...


In [238]:
def get_all_park_codes(df_historic_codes, df_units, df_visits, df_api_codes):

    # take the three historic-code-like columns, stack them into one Series
    historic = (
        df_historic_codes[["historic_code", "former_code", "alternate_code"]]
        .stack()
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    official = (
        df_units["units_code"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    visits = (
        df_visits["visits_code"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    api = (
        df_api_codes["api_code"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    #combine, dedupe, and sort
    all_codes = sorted(set(historic) | set(official) | set(visits) | set(api))
    return all_codes 

all_codes = get_all_park_codes(df_historic_codes, df_units, df_visits, df_api_codes)

len(all_codes), all_codes[:10]


(588,
 ['abli',
  'acad',
  'adam',
  'afam',
  'afbg',
  'agfo',
  'akro',
  'alag',
  'alca',
  'aleu'])

In [239]:
def add_park_codes(
    df_mortality_incidents,
    df_historic_codes,
    df_units,
    df_visits,
    df_api_codes
):
    out = df_mortality_incidents.copy()
    
    #Historic
    hist_map = (
        df_historic_codes
        .dropna(subset=["historic_name", "historic_code"])
        .drop_duplicates(subset="historic_name")
        .set_index("historic_name")["historic_code"]
    )
    out["historic_code"] = out["mortality_name"].map(hist_map)
    
    #Units
    units_map = (
        df_units
        .dropna(subset=["units_name", "units_code"])
        .drop_duplicates(subset="units_name")
        .set_index("units_name")["units_code"]
    )
    out["units_code"] = out["mortality_name"].map(units_map)
    
    #Visits
    visit_map = (
        df_visits
        .dropna(subset=["visits_name", "visits_code"])
        .drop_duplicates(subset="visits_name")
        .set_index("visits_name")["visits_code"]
    )
    out["visits_code"] = out["mortality_name"].map(visit_map)

    #API
    api_map = (
        df_api_codes
        .dropna(subset=["api_name", "api_code"])
        .drop_duplicates(subset="api_name")
        .set_index("api_name")["api_code"]
    )
    out["api_code"] = out["mortality_name"].map(visit_map)
    
    return out


#Call
df_with_codes = add_park_codes(
    df_mortality_incidents=df_mortality_incidents,
    df_historic_codes=df_historic_codes,
    df_units=df_units,
    df_visits=df_visits,
    df_api_codes=df_api_codes
)

df_with_codes.head()

,incident_id,park_id,mortality_name,original_name,incident_date,date_id,cause_id,intent_id,sex_id,activity_id,age_range_original,age_min,age_max,historic_code,units_code,visits_code,api_code
0,1,82,glen canyon national recreation area,Glen Canyon National Recreation Area,2007-01-01,1,39,3,2,20,65+,65.0,NaN,glca,NaN,NaN,NaN
1,2,84,golden gate national recreation area,Golden Gate National Recreation Area,2007-01-22,22,7,4,2,37,Unknown,NaN,NaN,goga,NaN,NaN,NaN
2,3,84,golden gate national recreation area,Golden Gate National Recreation Area,2007-01-22,22,39,3,2,37,Unknown,NaN,NaN,goga,NaN,NaN,NaN
3,4,139,natchez trace parkway,Natchez Trace Parkway,2007-01-29,29,24,4,1,11,15-24,15.0,24.0,natr,natr,NaN,NaN
4,5,139,natchez trace parkway,Natchez Trace Parkway,2007-01-29,29,24,4,1,11,45-54,45.0,54.0,natr,natr,NaN,NaN


In [240]:
df_with_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4635 entries, 0 to 4634
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   incident_id         4635 non-null   int64         
 1   park_id             4635 non-null   int64         
 2   mortality_name      4635 non-null   object        
 3   original_name       4635 non-null   object        
 4   incident_date       4635 non-null   datetime64[ns]
 5   date_id             4635 non-null   int64         
 6   cause_id            4635 non-null   int64         
 7   intent_id           4635 non-null   int64         
 8   sex_id              4635 non-null   int64         
 9   activity_id         4635 non-null   int64         
 10  age_range_original  4635 non-null   object        
 11  age_min             3992 non-null   float64       
 12  age_max             3165 non-null   float64       
 13  historic_code       4375 non-null   object      

In [241]:
def get_parks_missing_codes(df_with_codes):

    code_columns = {
        "historic": "historic_code",
        "units": "units_name",
        "visit": "visits_code",
        "api": "api_code"
    }

    missing = {}

    for label, col in code_columns.items():
        parks = (
            df_with_codes[df_with_codes[col].isna()]["mortality_name"]
            .dropna()
            .unique()
        )
        missing[label] = sorted(parks)

    return missing


In [242]:
missing_all = df_with_codes[
    df_with_codes["historic_code"].isna()
    & df_with_codes["units_code"].isna()
    & df_with_codes["visits_code"].isna()
]

missing_all.head(1)

,incident_id,park_id,mortality_name,original_name,incident_date,date_id,cause_id,intent_id,sex_id,activity_id,age_range_original,age_min,age_max,historic_code,units_code,visits_code,api_code
29,30,54,denali national park and preserve,Denali National Park & Preserve,2007-04-23,113,9,4,1,6,35-44,35.0,44.0,NaN,NaN,NaN,NaN


In [243]:
missing_all_parks = missing_all["mortality_name"].unique()
missing_all_parks


array(['denali national park and preserve',
       'chickasaw national recreation area', 'suitland',
       'indiana dunes national park', 'hawaii volcanoes national park',
       'white sands national park', 'dry tortugas national park',
       'wrangell - st elias national park and reserve',
       'home of franklin d roosevelt national historic site',
       'lyndon b johnson national historical park',
       'golden spike national historical park',
       'new river gorge national park and preserve', 'not reported',
       'yorktown battlefield part of colonial national historical park'],
      dtype=object)

I will not be able to match Suitland, as there are several potential park units that may be refered to as Suitland. The other missing item - "wrangell - st elias national park and reserve" is actully a typo. I'll corect that and then there should be a direct match.

In [244]:
#Correct name in base df
df["mortality_name"] = df["mortality_name"].replace(
    "wrangell - st elias national park and reserve",
    "wrangell - st elias national park and preserve"
)

#Rebuild df_with_codes
df_with_codes = add_park_codes(
    df,
    df_historic_codes,
    df_units,
    df_visits,
    df_api_codes
)

mask = df_with_codes["mortality_name"] == "wrangell - st elias national park and preserve"

df_with_codes.loc[mask, "visits_code"] = "wrst"

In [245]:
missing_all = df_with_codes[
    df_with_codes["historic_code"].isna()
    & df_with_codes["units_code"].isna()
    & df_with_codes["visits_code"].isna()
]

missing_all_parks = missing_all["mortality_name"].unique()
missing_all_parks

array(['denali national park and preserve',
       'chickasaw national recreation area', 'suitland',
       'indiana dunes national park', 'hawaii volcanoes national park',
       'white sands national park', 'dry tortugas national park',
       'home of franklin d roosevelt national historic site',
       'lyndon b johnson national historical park',
       'golden spike national historical park',
       'new river gorge national park and preserve', 'not reported',
       'yorktown battlefield part of colonial national historical park'],
      dtype=object)

I've obtained all the data I need, and now I want to create a master key of park codes and names that I can use for analysis.

I'm going to create a map to rename the columns I'll be pulling in from the df_with_Codes, since I've already consolidated some there. Then I'll create the df that will hold all my unit code keys.

In [246]:
df_with_codes.columns

Index(['incident_date', 'mortality_name', 'cause', 'cause_group', 'intent',
       'sex', 'age_range', 'activity', 'original_name', 'age_range_min',
       'age_range_max', 'month', 'year', 'historic_code', 'units_code',
       'visits_code', 'api_code'],
      dtype='object')

In [247]:
key_map = {
    "mortality_name": "mortality_name",
    "historic_code": "historic_code",
    "api_code": "api_code",
    "units_code": "units_code",
    "visits_code": "visits_code",
}

df_code_key = (
    #select only the columns in key_may
    df_with_codes[list(key_map.keys())]
    #rename
    .rename(columns=key_map)
    .drop_duplicates()
    .reset_index(drop=True)
)

df_code_key.head(1)

,mortality_name,historic_code,api_code,units_code,visits_code
0,glen canyon national recreation area,glca,NaN,NaN,NaN


I'm going to use my all_codes list to create a df that I can use to hold all parks

In [248]:
df_all_codes = pd.DataFrame({
    'park_id': range(1, len(all_codes) + 1),
    'key_code': all_codes
})

df_all_codes.head()

,park_id,key_code
0,1,abli
1,2,acad
2,3,adam
3,4,afam
4,5,afbg


Before I join, I want to create a helper column with the best code match for a join. I'm going to use the visits code as the priority code, because the analysis I'm doing will be normalized for visitation and I want to make sure that's my primary match.

In [249]:
df_code_key["best_code"] = (
    df_code_key["visits_code"]
    .fillna(df_code_key["api_code"])
    .fillna(df_code_key["units_code"])
    .fillna(df_code_key["historic_code"])
)

df_code_key.head()

,mortality_name,historic_code,api_code,units_code,visits_code,best_code
0,glen canyon national recreation area,glca,NaN,NaN,NaN,glca
1,golden gate national recreation area,goga,NaN,NaN,NaN,goga
2,natchez trace parkway,natr,NaN,natr,NaN,natr
3,gateway national recreation area,gate,NaN,NaN,NaN,gate
4,black canyon of the gunnison national park,blca,NaN,NaN,NaN,blca


In [250]:
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   mortality_name  203 non-null    object
 1   historic_code   187 non-null    object
 2   api_code        6 non-null      object
 3   units_code      9 non-null      object
 4   visits_code     7 non-null      object
 5   best_code       190 non-null    object
dtypes: object(6)
memory usage: 9.6+ KB


In [251]:
df_code_key.loc[df_code_key["best_code"].isna(), "mortality_name"]

18                     denali national park and preserve
27                    chickasaw national recreation area
55                                              suitland
62                           indiana dunes national park
72                        hawaii volcanoes national park
93                             white sands national park
126                           dry tortugas national park
136    home of franklin d roosevelt national historic...
156            lyndon b johnson national historical park
179                golden spike national historical park
187           new river gorge national park and preserve
190                                         not reported
200    yorktown battlefield part of colonial national...
Name: mortality_name, dtype: object

I utilized Claude to create a manual map to assign park codes that are correct to names that do not yet have a code. I manually checked each code to ensure it was accurate and appropriately matched.

In [252]:
# Manual mapping for missing park codes
missing_park_map = {
    'denali national park and preserve': 'dena',
    'chickasaw national recreation area': 'chic',
    'suitland': None,  # Not a park - likely admin location
    'indiana dunes national park': 'indu',
    'hawaii volcanoes national park': 'havo',
    'white sands national park': 'whsa',
    'dry tortugas national park': 'drto',
    'home of franklin d roosevelt national historic site': 'hofr',
    'lyndon b johnson national historical park': 'lyjo',
    'golden spike national historical park': 'gosp',
    'new river gorge national park and preserve': 'neri',
    'not reported': np.nan,  # No park
    'yorktown battlefield part of colonial national historical park': 'colo'
}

# Verify these codes exist in visits dataset
visits_codes = df_visits['visits_code'].dropna().str.lower().unique()
for name, code in missing_park_map.items():
    if code is not None and pd.notna(code):
        exists = code in visits_codes
        print(f"{code:6s} - {name[:50]:50s} {'✓' if exists else '✗ NOT IN VISITS'}")
    else:
        print(f"{'NULL':6s} - {name[:50]:50s} (intentionally NULL)")

dena   - denali national park and preserve                  ✓
chic   - chickasaw national recreation area                 ✓
NULL   - suitland                                           (intentionally NULL)
indu   - indiana dunes national park                        ✓
havo   - hawaii volcanoes national park                     ✓
whsa   - white sands national park                          ✓
drto   - dry tortugas national park                         ✓
hofr   - home of franklin d roosevelt national historic sit ✓
lyjo   - lyndon b johnson national historical park          ✓
gosp   - golden spike national historical park              ✓
neri   - new river gorge national park and preserve         ✓
NULL   - not reported                                       (intentionally NULL)
colo   - yorktown battlefield part of colonial national his ✓


In [ ]:
# Add a column with the mapped codes
df_code_key['mapped_code'] = df_code_key['mortality_name'].map(missing_park_map)

# Fill in best_code where missing using the manual mapping
df_code_key['best_code'] = df_code_key['best_code'].fillna(df_code_key['mapped_code'])

# Drop the temporary column
df_code_key = df_code_key.drop('mapped_code', axis=1)

# Check results
df_code_key[df_code_key['mortality_name'].isin(missing_park_map.keys())][['mortality_name', 'best_code']].drop_duplicates()

Still missing codes: 2


,mortality_name,best_code
18,denali national park and preserve,dena
27,chickasaw national recreation area,chic
55,suitland,None
62,indiana dunes national park,indu
72,hawaii volcanoes national park,havo
93,white sands national park,whsa
126,dry tortugas national park,drto
136,home of franklin d roosevelt national historic...,hofr
156,lyndon b johnson national historical park,lyjo
179,golden spike national historical park,gosp


In [254]:
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   mortality_name  203 non-null    object
 1   historic_code   187 non-null    object
 2   api_code        6 non-null      object
 3   units_code      9 non-null      object
 4   visits_code     7 non-null      object
 5   best_code       201 non-null    object
dtypes: object(6)
memory usage: 9.6+ KB


I'm going to drop the rows that have a mortality_name that can't be matched to any code because those won't go in my key.

In [255]:
df_code_key = df_code_key[df_code_key["best_code"].notna()].reset_index(drop=True)

df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   mortality_name  201 non-null    object
 1   historic_code   187 non-null    object
 2   api_code        6 non-null      object
 3   units_code      9 non-null      object
 4   visits_code     7 non-null      object
 5   best_code       201 non-null    object
dtypes: object(6)
memory usage: 9.6+ KB


In [256]:
df_all_codes.head(1)

,park_id,key_code
0,1,abli


In [257]:
df_code_key = df_all_codes.merge(
    df_code_key,
    how="left",
    left_on="key_code",
    right_on="best_code"
)

df_code_key.head(1)

,park_id,key_code,mortality_name,historic_code,api_code,units_code,visits_code,best_code
0,1,abli,NaN,NaN,NaN,NaN,NaN,NaN


In [258]:
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   park_id         590 non-null    int64 
 1   key_code        590 non-null    object
 2   mortality_name  201 non-null    object
 3   historic_code   187 non-null    object
 4   api_code        6 non-null      object
 5   units_code      9 non-null      object
 6   visits_code     7 non-null      object
 7   best_code       201 non-null    object
dtypes: int64(1), object(7)
memory usage: 37.0+ KB


In [259]:
df_code_key[df_code_key["key_code"].duplicated(keep=False)]

,park_id,key_code,mortality_name,historic_code,api_code,units_code,visits_code,best_code
122,123,colo,colonial national historical park,colo,NaN,NaN,NaN,colo
123,123,colo,yorktown battlefield part of colonial national...,NaN,NaN,NaN,NaN,colo
390,390,neri,new river gorge national river,neri,NaN,NaN,NaN,neri
391,390,neri,new river gorge national park and preserve,NaN,NaN,NaN,NaN,neri


In [260]:
#Primary park units table
df_park_units = df_code_key[['park_id', 'key_code']].drop_duplicates(subset='park_id')

df_park_units.head()

,park_id,key_code
0,1,abli
1,2,acad
2,3,adam
3,4,afam
4,5,afbg


In [261]:
def safe_merge_fill(
    left,
    right,
    left_key="key_code",
    right_key=None,
    cols_map=None,
    keep="first",
    keep_right_key=True,
):
    left = left.copy()
    
    #Build a list of columns needed from right
    right_cols = [right_key] + list(cols_map.keys())
    
    #Dedupe park code rows in df to be merged
    right_unique = (
        right[right_cols]
        .dropna(subset=[right_key])
        .drop_duplicates(subset=[right_key], keep=keep)
    )
    
    #Add helper names
    rename_dict = {rcol: f"__tmp__{lcol}" for rcol, lcol in cols_map.items()}
    #Rename join key to match
    rename_dict[right_key] = left_key
    
    #Create helper column for right values
    if keep_right_key and right_key not in cols_map:
        rename_dict[right_key] = f"__merge_key__"
    
    right_unique = right_unique.rename(columns=rename_dict)
    
    #Merge on left_key
    if keep_right_key and right_key not in cols_map:
        #Merge using the temp merge key
        merged = left.merge(
            right_unique,
            how="left",
            left_on=left_key,
            right_on="__merge_key__",
        )
        #Fill right_key column from merge key
        if right_key in merged.columns:
            merged[right_key] = merged[right_key].fillna(merged["__merge_key__"]).infer_objects(copy=False)
        else:
            merged[right_key] = merged["__merge_key__"]
        #Drop the temp merge key
        merged = merged.drop(columns=["__merge_key__"])
    else:
        #Standard merge when right_key is in cols_map or not needed
        merged = left.merge(
            right_unique,
            how="left",
            on=left_key,
        )
    
    #Fill left columns from helper columns, then drop helpers
    for rcol, lcol in cols_map.items():
        src = f"__tmp__{lcol}"
        if src in merged.columns:
            if lcol in merged.columns:
                merged[lcol] = merged[lcol].fillna(merged[src]).infer_objects(copy=False)
            else:
                merged[lcol] = merged[src]
            merged = merged.drop(columns=[src])
    
    return merged

In [262]:
df_code_key = safe_merge_fill(
    left=df_code_key,
    right=df_units,
    left_key="key_code",
    right_key="units_code",
    cols_map={
        "units_name": "units_name",
    },
    keep_right_key=True,
)

df_code_key.head(1)

,park_id,key_code,mortality_name,historic_code,api_code,units_code,visits_code,best_code,units_name
0,1,abli,NaN,NaN,NaN,abli,NaN,NaN,abraham lincoln birthplace


In [263]:
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   park_id         590 non-null    int64 
 1   key_code        590 non-null    object
 2   mortality_name  201 non-null    object
 3   historic_code   187 non-null    object
 4   api_code        6 non-null      object
 5   units_code      409 non-null    object
 6   visits_code     7 non-null      object
 7   best_code       201 non-null    object
 8   units_name      409 non-null    object
dtypes: int64(1), object(8)
memory usage: 41.6+ KB


In [264]:
#visits
df_code_key = safe_merge_fill(
    left=df_code_key,
    right=df_visits,
    left_key="key_code",
    right_key="visits_code",
    cols_map={
        "visits_name": "visits_name",
    },
)

df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   park_id         590 non-null    int64 
 1   key_code        590 non-null    object
 2   mortality_name  201 non-null    object
 3   historic_code   187 non-null    object
 4   api_code        6 non-null      object
 5   units_code      409 non-null    object
 6   visits_code     400 non-null    object
 7   best_code       201 non-null    object
 8   units_name      409 non-null    object
 9   visits_name     400 non-null    object
dtypes: int64(1), object(9)
memory usage: 46.2+ KB


In [265]:
#api
df_code_key = safe_merge_fill(
    left=df_code_key,
    right=df_api_codes,
    left_key="key_code",
    right_key="api_code",
    cols_map={
        "api_name": "api_name",
    },
)
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   park_id         590 non-null    int64 
 1   key_code        590 non-null    object
 2   mortality_name  201 non-null    object
 3   historic_code   187 non-null    object
 4   api_code        476 non-null    object
 5   units_code      409 non-null    object
 6   visits_code     400 non-null    object
 7   best_code       201 non-null    object
 8   units_name      409 non-null    object
 9   visits_name     400 non-null    object
 10  api_name        476 non-null    object
dtypes: int64(1), object(10)
memory usage: 50.8+ KB


In [266]:
#Only look at alternateName = False rows
update_mask = df_historic_codes['alternate_name'] == False

#Create a temporary df with only those rows
df_to_update = df_historic_codes[update_mask].copy()

In [267]:
#sanity check on code that I know has multiple rows
df_to_update[df_to_update["historic_code"] == "colo"]

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code
118,Colonial National Historical Park,colo,False,<NA>,colonial national historical park,NaN,NaN


In [268]:
df_code_key = safe_merge_fill(
    left=df_code_key,
    right=df_to_update,
    left_key="key_code",
    right_key="historic_code",
    cols_map={
        "historic_name": "historic_name",
        "former_code": "former_code",
        "alternate_code": "alternate_code",
    },
)

df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   park_id         590 non-null    int64 
 1   key_code        590 non-null    object
 2   mortality_name  201 non-null    object
 3   historic_code   558 non-null    object
 4   api_code        476 non-null    object
 5   units_code      409 non-null    object
 6   visits_code     400 non-null    object
 7   best_code       201 non-null    object
 8   units_name      409 non-null    object
 9   visits_name     400 non-null    object
 10  api_name        476 non-null    object
 11  historic_name   558 non-null    object
 12  former_code     3 non-null      object
 13  alternate_code  1 non-null      object
dtypes: int64(1), object(13)
memory usage: 64.7+ KB


In [269]:
df_code_key = df_code_key.drop(columns=["best_code", "key_code", "park_id"])

df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   mortality_name  201 non-null    object
 1   historic_code   558 non-null    object
 2   api_code        476 non-null    object
 3   units_code      409 non-null    object
 4   visits_code     400 non-null    object
 5   units_name      409 non-null    object
 6   visits_name     400 non-null    object
 7   api_name        476 non-null    object
 8   historic_name   558 non-null    object
 9   former_code     3 non-null      object
 10  alternate_code  1 non-null      object
dtypes: object(11)
memory usage: 50.8+ KB


In [270]:
# Create park_id mapping from df_park_units
park_id_map = df_park_units.set_index('key_code')['park_id'].to_dict()

In [273]:
df_code_key.head()

,mortality_name,historic_code,api_code,units_code,visits_code,units_name,visits_name,api_name,historic_name,former_code,alternate_code
0,NaN,abli,abli,abli,abli,abraham lincoln birthplace,abraham lincoln birthplace nhp,abraham lincoln birthplace national historical...,abraham lincoln birthplace national historical...,NaN,NaN
1,acadia national park,acad,acad,acad,acad,acadia,acadia np,acadia national park,acadia national park,NaN,NaN
2,NaN,adam,adam,adam,adam,adams,adams nhp,adams national historical park,adams national historical park,NaN,NaN
3,NaN,afam,afam,NaN,NaN,NaN,NaN,african american civil war memorial,african american civil war memorial,NaN,NaN
4,NaN,afbg,afbg,afbg,afbg,african burial ground,african burial ground nm,african burial ground national monument,african burial ground national monument,NaN,NaN


In [275]:
#create key_code from best available code
df_code_key['key_code'] = (
    df_code_key['visits_code']
    .fillna(df_code_key['api_code'])
    .fillna(df_code_key['units_code'])
    .fillna(df_code_key['historic_code'])
)

# Now add park_id
df_code_key['park_id'] = df_code_key['key_code'].map(park_id_map)

In [276]:
df_code_key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mortality_name  201 non-null    object 
 1   historic_code   558 non-null    object 
 2   api_code        476 non-null    object 
 3   units_code      409 non-null    object 
 4   visits_code     400 non-null    object 
 5   units_name      409 non-null    object 
 6   visits_name     400 non-null    object 
 7   api_name        476 non-null    object 
 8   historic_name   558 non-null    object 
 9   former_code     3 non-null      object 
 10  alternate_code  1 non-null      object 
 11  key_code        587 non-null    object 
 12  park_id         587 non-null    float64
dtypes: float64(1), object(12)
memory usage: 60.1+ KB


I'll add to everything else.

In [277]:
#Map mortality_name → key_code first, then to park_id
mortality_to_code = df_code_key.dropna(subset=['mortality_name']).set_index('mortality_name')['key_code'].to_dict()
df_mortality_incidents['key_code'] = df_mortality_incidents['mortality_name'].map(mortality_to_code)
df_mortality_incidents['park_id'] = df_mortality_incidents['key_code'].map(park_id_map)

#Drop key_code
df_mortality_incidents = df_mortality_incidents.drop('key_code', axis=1)

In [278]:
df_mortality_incidents.head(1)

,incident_id,park_id,mortality_name,original_name,incident_date,date_id,cause_id,intent_id,sex_id,activity_id,age_range_original,age_min,age_max
0,1,219.0,glen canyon national recreation area,Glen Canyon National Recreation Area,2007-01-01,1,39,3,2,20,65+,65.0,NaN


In [279]:
# visits_code is already a park code
df_visits['park_id'] = df_visits['visits_code'].map(park_id_map)

df_visits.head(1)

,month,nonrecreation_visitors,recreation_visitors,visits_code,visits_name,year,name,park_id
0,1,0,4960,abli,abraham lincoln birthplace nhp,2007,Abraham Lincoln Birthplace NHP,1


In [280]:
#units
df_units['park_id'] = df_units['units_code'].map(park_id_map)

df_units.head(1)

,units_name,type,units_code,org_code,region,park_id
1,abraham lincoln birthplace,nhp,abli,5540,ser,1.0


In [281]:
# historic
df_historic_codes['park_id'] = df_historic_codes['historic_code'].map(park_id_map)

df_historic_codes.head(1)

,name,historic_code,alternate_name,region,historic_name,former_code,alternate_code,park_id
0,Abraham Lincoln Birthplace National Historical...,abli,False,<NA>,abraham lincoln birthplace national historical...,NaN,NaN,1.0


I'll obtain data about all activities are possible in the parks, as well as match any activities that are also listed in my df. This information will be downloaded from NPS API and documentation for that service can be found at https://www.nps.gov/subjects/developer/api-documentation.htm#/activities/parks/getActvitiesParks

In [282]:
API = "https://developer.nps.gov/api/v1/activities/parks"
HEADERS = {"X-Api-Key": API_KEY}
LIMIT = 500

In [283]:
all_codes = [c.upper() for c in all_codes]

In [284]:
def pages():
    start = 0
    while True:
        r = requests.get(API, headers=HEADERS, params={"limit": LIMIT, "start": start})
        r.raise_for_status()
        payload = r.json()
        data = payload.get("data", [])
        if not data:
            break
        yield from data
        total = int(payload.get("total", 0))
        start += LIMIT
        if start >= total:
            break

park_activities = {}
for item in pages():
    name = item.get("name")
    parks = item.get("parks", [])
    if name:
        park_activities.setdefault(name, []).extend(
            {"activities_code": p.get("parkCode"), "states": p.get("states"), "activities_name": p.get("fullName")}
            for p in parks)

In [285]:
rows = [
    {"activity": activity, **park}
    for activity, parks in park_activities.items()
    for park in parks
]

df_park_activities = pd.DataFrame(rows)
df_park_activities.head()

,activity,activities_code,states,activities_name
0,Arts and Culture,acad,ME,Acadia National Park
1,Arts and Culture,afbg,NY,African Burial Ground National Monument
2,Arts and Culture,agfo,NE,Agate Fossil Beds National Monument
3,Arts and Culture,alfl,TX,Alibates Flint Quarries National Monument
4,Arts and Culture,alka,HI,Ala Kahakai National Historic Trail


In [286]:
df_park_activities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4051 entries, 0 to 4050
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   activity         4051 non-null   object
 1   activities_code  4051 non-null   object
 2   states           4051 non-null   object
 3   activities_name  4051 non-null   object
dtypes: object(4)
memory usage: 126.7+ KB


In [287]:
cols = ["activity", 
        "activities_code", 
        "states", 
        "activities_name"]

df_park_activities[cols] = df_park_activities[cols].apply(clean)

In [288]:
df_park_activities.to_csv("../data/park_activities.csv", index=False)

In [289]:
df_park_activities["activity"].unique()

array(['arts and culture', 'astronomy', 'auto and atv', 'biking',
       'boating', 'camping', 'canyoneering', 'caving', 'climbing',
       'compass and gps', 'dog sledding', 'fishing', 'flying', 'food',
       'golf', 'guided tours', 'hands-on', 'hiking', 'horse trekking',
       'hunting and gathering', 'ice skating', 'junior ranger program',
       'living history', 'museum exhibits', 'paddling', 'park film',
       'playground', 'scuba diving', 'shopping', 'skiing', 'snorkeling',
       'snow play', 'snowmobiling', 'snowshoeing', 'surfing', 'swimming',
       'team sports', 'tubing', 'water skiing', 'wildlife watching'],
      dtype=object)

In [290]:
df_activity_types.columns

Index(['activity_id', 'activity_name'], dtype='object')

In [291]:
df_activity_types["activity_name"].unique()

array(['base jumping', 'bicycling', 'camping', 'canyoneering', 'caving',
       'climbing', 'crossing river', 'diving (land)', 'diving - free',
       'diving - scuba', 'driving', 'eating', 'fishing', 'flying',
       'hiking', 'homicide', 'horseback (or mule) riding', 'hunting',
       'illegal activity', 'not reported', 'other', 'photographing',
       'playing sports', 'rock scrambeling', 'rock scrambling', 'sitting',
       'skateboarding', 'skiing', 'sleeping', 'snorkeling',
       'snowboarding', 'snowmobiling', 'snowshoeing', 'suicide',
       'swimming', 'undetermined', 'vessel related', 'wading', 'walking',
       'wildlife watching'], dtype=object)

I'm going to match the ones that can be matched.

In [292]:
#manual map for observable matches
manual_map = {
    # Matches
    'vessel related': 'boating',
    'driving': 'auto and atv',
    'diving - scuba': 'scuba diving',
    'horseback (or mule) riding': 'horse trekking',
    'bicycling': 'biking',
    'playing sports': 'team sports',
    'hunting': 'hunting and gathering',
    'diving - free': 'swimming',
    'rock scrambling': 'climbing',
    'snowboarding': 'skiing',
    'wading': 'hiking',
    'crossing river': 'hiking',
    # No Matches
    'not reported': np.nan,
    'other': np.nan,
    'photographing': np.nan,
    'walking': np.nan,
    'suicide': np.nan,
    'diving (land)': np.nan,
    'eating': np.nan,
    'sleeping': np.nan,
    'base jumping': np.nan,
    'skateboarding': np.nan,
    'homicide': np.nan,
    'illegal activity': np.nan,
    'sitting': np.nan,
    'undetermined': np.nan
}

#create map
df_activity_map = pd.DataFrame(
    list(manual_map.items()), 
    columns=["activity_name", "park_activity"]
)

df_activity_map.head(1)

,activity_name,park_activity
0,vessel related,boating


In [293]:
#Begin merge with only mortality activities
df_all_activities = df_activity_types[['activity_id', 'activity_name']].copy()

#Merge manual map
df_all_activities = df_all_activities.merge(
    df_activity_map,
    on="activity_name",
    how="left"
)

#find umapped park activities
park_activities_list = df_park_activities["activity"].unique()
existing_park_activities = df_all_activities['park_activity'].dropna().unique()

#park activities that don't have a match
missing_park_activities = [
    pa for pa in park_activities_list 
    if pa not in existing_park_activities
]

# Add them as new rows where activity_name = park_activity
next_id = df_all_activities['activity_id'].max() + 1
new_activities = pd.DataFrame({
    'activity_id': range(next_id, next_id + len(missing_park_activities)),
    'activity_name': np.nan,
    'park_activity': missing_park_activities
})

# Combine
df_activity_types = pd.concat(
    [df_all_activities, new_activities],
    ignore_index=True
).sort_values('activity_id').reset_index(drop=True)

df_activity_types.head(1)

,activity_id,activity_name,park_activity
0,1,base jumping,NaN


In [294]:
df_activity_types.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   activity_id    69 non-null     int64 
 1   activity_name  40 non-null     object
 2   park_activity  41 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.7+ KB


In [295]:
df_park_activities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4051 entries, 0 to 4050
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   activity         4051 non-null   object
 1   activities_code  4051 non-null   object
 2   states           4051 non-null   object
 3   activities_name  4051 non-null   object
dtypes: object(4)
memory usage: 126.7+ KB


Now I'll add the activity_id to df_park_activies to get it ready to merge.

In [296]:
# Create a mapping from park_activity to activity_id
activity_map = df_activity_types[['activity_id', 'park_activity']].dropna(subset=['park_activity'])

# Merge to add activity_id to df_park_activities
df_park_activities = df_park_activities.merge(
    activity_map,
    left_on='activity', 
    right_on='park_activity',
    how='left'
).drop('park_activity', axis=1)

df_park_activities.head(1)

,activity,activities_code,states,activities_name,activity_id
0,arts and culture,acad,me,acadia national park,41


I'll make sure everything is mapped.

In [297]:
unmapped = df_park_activities[df_park_activities['activity_id'].isna()]['activity'].unique()
unmapped

array([], dtype=object)

It is, so I'll drop the column.

In [298]:
df_park_activities = df_park_activities.drop('activity', axis=1)

In [299]:
df_park_activities.head(1)

,activities_code,states,activities_name,activity_id
0,acad,me,acadia national park,41


In [300]:
df_park_activities['park_id'] = df_park_activities['activities_code'].map(park_id_map)

df_park_activities.head(1)

,activities_code,states,activities_name,activity_id,park_id
0,acad,me,acadia national park,41,2.0


In [301]:
#df_park_activities = df_park_activities.drop(columns=['activities_code', 'activities_name'])

df_park_activities.head(1)

,activities_code,states,activities_name,activity_id,park_id
0,acad,me,acadia national park,41,2.0


Now, we'll obtain the amenities data to be used to determine if there is any correlation between available amenities and fatal events. I'll start by testing the API endpoint.

In [302]:
BASE = "https://developer.nps.gov/api/v1"


In [303]:
r = requests.get(
    "https://developer.nps.gov/api/v1/amenities",
    headers={"X-Api-Key": API_KEY},
    params={"limit": 1}
)
print(r.status_code)
print(r.json().get("total"))

200
127


I want to obtain all the data from the amenities API, so I'm going to define a fetch_all function that will continue calling until I've downloaded everything.

In [304]:
#return 200 results at a time, pause between pages
def fetch_all(endpoint, params=None, page_size=200, sleep=0.1):
    #create empty params dictionary
    params = dict(params or {})
    params.setdefault("limit", page_size)
    start = 0
    out = []
    while True:
        params["start"] = start
        r = requests.get(f"{BASE}/{endpoint}", headers=HEADERS, params=params, timeout=30)
        #429 = too many requests, so pause and then resume if this response is received
        if r.status_code == 429:
            time.sleep(1.0); continue
        #Throw an exception for any other errors
        r.raise_for_status()
        payload = r.json()
        data = payload.get("data", [])
        out.extend(data)
        #get total number of records
        total = int(payload.get("total", len(out)))
        #move start forward for next call
        start += page_size
        #stop if we reach the total number of items or receive an empty response
        if start >= total or not data:
            break
        #pause between pages
        time.sleep(sleep)
    return out

The amenities API containes many nested lists, so I'll tell python how to handle that. During the trial and error of crafting the call, it because apparent that sometimes there are dictionaries nested in several layers of lists, so the API call needs to be able to find all the data.

In [305]:
def _iter_records(maybe):
    #If empty, stop
    if maybe is None:
        return
    #if it's a dictionary, yiled the dictionary 
    if isinstance(maybe, dict):
        yield maybe
    #if it's a list, look at each item    
    elif isinstance(maybe, list):
        for item in maybe:
            #if item is dictionary, yeld the dictionary
            if isinstance(item, dict):
                yield item
            #if it's list, look at each item    
            elif isinstance(item, list):
                for sub in item:
                    if isinstance(sub, dict):
                        yield sub

Keys also sometimes have different names

In [306]:
def _get(rec, *keys, default=None):
    
    for k in keys:
    
        #only proceed if dictionary
        if isinstance(rec, dict) and k in rec:
            
            #return the first key that is a match
            return rec[k]
    
    #return None if no matches
    return default

In [307]:
#Fetch all amenities
amenities = fetch_all("amenities")

#build map of all amenities
amenity_map = {a.get("id"): a.get("name") for a in amenities}

In [308]:
rows = []

#loop through every amenity returned
for a_id, a_name in amenity_map.items():
    
    #fetch all places that have this amenity
    data_p = fetch_all("amenities/parksplaces", params={"id": a_id})
    
    #loop through all records in the response
    for rec in _iter_records(data_p):
        
        #Places

        #The API can return 2 different structures
        #{'parksPlaces': [...]}; 
        parks_places = _get(rec, "parksPlaces")
         #{'parks':[{'parkCode', 'places':[...]}, ...]}
        parks_container = _get(rec, "parks")

        #{'parksPlaces': [...]}; 
        if isinstance(parks_places, list):
            for pl in _iter_records(parks_places):
                rows.append({
                    "parkCode": _get(pl, "parkCode"),
                    "assetType": "place",
                    "assetId": _get(pl, "id"),
                    "assetName": _get(pl, "name", "title"),
                    "amenityId": a_id,
                    "amenityName": a_name
                })

        # #{'parks':[{'parkCode', 'places':[...]}, ...]}
        if isinstance(parks_container, list):
            for p in _iter_records(parks_container):
                pc = _get(p, "parkCode")
                for pl in _iter_records(_get(p, "places")):
                    rows.append({
                        "parkCode": pc,
                        "assetType": "place",
                        "assetId": _get(pl, "id"),
                        "assetName": _get(pl, "name", "title"),
                        "amenityId": a_id,
                        "amenityName": a_name
                    })

        # Visitor Centers
        data_vc = fetch_all("amenities/parksvisitorcenters", params={"id": a_id})
        
        for rec in _iter_records(data_vc):
            
            #{'parksVisitorCenters': [...]}
            pvc = _get(rec, "parksVisitorCenters")
            
            #{'parks':[{'parkCode','visitorCenters':[...]}]}
            parks_container = _get(rec, "parks")

            #{'parksVisitorCenters': [...]}
            if isinstance(pvc, list):
                for vc in _iter_records(pvc):
                    rows.append({
                        "parkCode": _get(vc, "parkCode"),
                        "assetType": "visitorCenter",
                        "assetId": _get(vc, "id"),
                        "assetName": _get(vc, "name", "title"),
                        "amenityId": a_id,
                        "amenityName": a_name
                    })

            #{'parks':[{'parkCode','visitorCenters':[...]}]}
            if isinstance(parks_container, list):
                for p in _iter_records(parks_container):
                    pc = _get(p, "parkCode")
                    for vc in _iter_records(_get(p, "visitorCenters")):
                        rows.append({
                            "parkCode": pc,
                            "assetType": "visitorCenter",
                            "assetId": _get(vc, "id"),
                            "assetName": _get(vc, "name", "title"),
                            "amenityId": a_id,
                            "amenityName": a_name
                        })

#create df
expected_cols = ["parkCode","assetType","assetId","assetName","amenityId","amenityName"]
df_amenities = pd.DataFrame(rows, columns=expected_cols)


In [309]:
df_amenities.head()

,parkCode,assetType,assetId,assetName,amenityId,amenityName
0,badl,place,5C566C36-F3C7-4CFA-A35C-9086329851B9,Cedar Pass Lodge,A1B0AD01-740C-41E7-8412-FBBEDD5F1443,ATM/Cash Machine
1,bibe,place,CD4C8274-1C47-4A75-BD3C-EA44406AADBE,Study Butte and Terlingua,A1B0AD01-740C-41E7-8412-FBBEDD5F1443,ATM/Cash Machine
2,bibe,place,0776DBE9-4A28-4DD1-A745-80D114FD09DE,Chisos Mountains Lodge,A1B0AD01-740C-41E7-8412-FBBEDD5F1443,ATM/Cash Machine
3,bibe,place,B2FEBCC4-F387-4297-97ED-83164844B502,Rio Grande Village Store,A1B0AD01-740C-41E7-8412-FBBEDD5F1443,ATM/Cash Machine
4,bibe,place,A216B05A-5D03-4849-A22E-2EBA088463BB,Panther Junction Service Station,A1B0AD01-740C-41E7-8412-FBBEDD5F1443,ATM/Cash Machine


In [310]:
df_amenities.info(
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47586 entries, 0 to 47585
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   parkCode     47586 non-null  object
 1   assetType    47586 non-null  object
 2   assetId      47586 non-null  object
 3   assetName    47586 non-null  object
 4   amenityId    47586 non-null  object
 5   amenityName  47586 non-null  object
dtypes: object(6)
memory usage: 2.2+ MB


In [311]:
df_amenities = df_amenities.sort_values("parkCode").reset_index(drop=True)

df_amenities.head()

,parkCode,assetType,assetId,assetName,amenityId,amenityName
0,abli,place,78ED430A-ECEF-435B-B8F6-25964AD607C2,The First Lincoln Memorial at Abraham Lincoln ...,EAB59B1F-2FA1-4545-A322-2C19149AF7FC,Wheelchair Accessible
1,abli,place,D5265572-4FD7-4078-A73E-F9B13956C5E5,Lincoln Tavern,4E4D076A-6866-46C8-A28B-A129E2B8F3DB,Accessible Rooms
2,abli,place,03F8157F-CA4F-49D2-9638-15A9B39CE3EB,Overlook Landing - Pathway of a President,EAB59B1F-2FA1-4545-A322-2C19149AF7FC,Wheelchair Accessible
3,abli,place,D5265572-4FD7-4078-A73E-F9B13956C5E5,Lincoln Tavern,EAB59B1F-2FA1-4545-A322-2C19149AF7FC,Wheelchair Accessible
4,abli,place,D5265572-4FD7-4078-A73E-F9B13956C5E5,Lincoln Tavern,08EDBF67-DF67-4D3B-83C1-8B2844228C4D,Water - Non-Potable


In [312]:
df_amenities = df_amenities.drop(columns=[
    "assetId",
    "amenityId"])

In [313]:
df_amenities = df_amenities.rename(columns={
    "parkCode": "amenities_code",
    "assetType": "type",
    "amenityName": "amenity_name",
    "assetName": "asset_name"
})

In [314]:
df_amenities["original_asset_name"] = df_amenities["asset_name"]

df_amenities.head(1)

,amenities_code,type,asset_name,amenity_name,original_asset_name
0,abli,place,The First Lincoln Memorial at Abraham Lincoln ...,Wheelchair Accessible,The First Lincoln Memorial at Abraham Lincoln ...


In [315]:
cols = ["amenities_code", 
        "asset_name", 
        "amenity_name"]

df_amenities[cols] = df_amenities[cols].apply(clean)

df_amenities.head(1)

,amenities_code,type,asset_name,amenity_name,original_asset_name
0,abli,place,the first lincoln memorial at abraham lincoln ...,wheelchair accessible,The First Lincoln Memorial at Abraham Lincoln ...


In [316]:
len(df_amenities["amenity_name"].unique())

125

In [317]:
len(df_amenities["amenities_code"].unique())

399

In [318]:
df_amenities.to_csv("../data/nps_amenities.csv", index=False)

I'm going to check to see if there are any new park codes in this dataset.

In [319]:
# Find missing parkCodes
missing_codes = (
    df_amenities["amenities_code"]
    .str.lower()
    .dropna()
    .loc[~df_amenities["amenities_code"].str.upper().isin(all_codes)]
    .unique()
)

# Add them only if they exist
if len(missing_codes) > 0:
    all_codes = list(set(all_codes).union(missing_codes))

len(missing_codes)


0

I'm going to add the park_id to amenities.

In [320]:
df_amenities['park_id'] = df_amenities['amenities_code'].map(park_id_map)

df_amenities.head(1)

,amenities_code,type,asset_name,amenity_name,original_asset_name,park_id
0,abli,place,the first lincoln memorial at abraham lincoln ...,wheelchair accessible,The First Lincoln Memorial at Abraham Lincoln ...,1


I'm going to prepare some additional dfs to make more efficient db tables.

In [321]:
unique_amenities = df_amenities['amenity_name'].dropna().unique()
df_amenity_types = pd.DataFrame({
    'amenity_id': range(1, len(unique_amenities) + 1),
    'amenity_name': sorted(unique_amenities)
})

df_amenity_types.head()

,amenity_id,amenity_name
0,1,accessible rooms
1,2,accessible sites
2,3,amphitheater
3,4,animal-safe food storage
4,5,aquatic invasive species inspection


In [322]:
#Add amenity_id
amenity_map = df_amenity_types.set_index('amenity_name')['amenity_id'].to_dict()
df_amenities['amenity_id'] = df_amenities['amenity_name'].map(amenity_map)

df_amenities.head(1)

,amenities_code,type,asset_name,amenity_name,original_asset_name,park_id,amenity_id
0,abli,place,the first lincoln memorial at abraham lincoln ...,wheelchair accessible,The First Lincoln Memorial at Abraham Lincoln ...,1,124


In [323]:
df_amenities = df_amenities.drop(columns=['amenities_code', 'type', 'original_asset_name'])

df_amenities.head(1)

,asset_name,amenity_name,park_id,amenity_id
0,the first lincoln memorial at abraham lincoln ...,wheelchair accessible,1,124


Next, I'll add a dataset with the geographic boundaries for all of the NPS park units. This file can be found at https://irma.nps.gov/DataStore/Reference/Profile/2224545?lnv=True, or in the nps_boundary folder of this project.

In [324]:
# Load the boundary layer
df_geo = gpd.read_file("../data/nps_boundary/nps_boundary.shp")

# Check what the unit-code column is actually called
print(df_geo.columns)
df_geo.head()

Index(['OBJECTID', 'UNIT_CODE', 'GIS_Notes', 'UNIT_NAME', 'DATE_EDIT', 'STATE',
       'REGION', 'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY', 'METADATA', 'PARKNAME',
       'CreationDa', 'Creator', 'EditDate', 'Editor', 'GlobalID', 'Shape_Leng',
       'Shape_Area', 'geometry'],
      dtype='object')


,OBJECTID,UNIT_CODE,GIS_Notes,UNIT_NAME,DATE_EDIT,STATE,REGION,GNIS_ID,UNIT_TYPE,CREATED_BY,METADATA,PARKNAME,CreationDa,Creator,EditDate,Editor,GlobalID,Shape_Leng,Shape_Area,geometry
0,1,MALL,Lands - http://landsnet.nps.gov/tractsnet/docu...,National Mall,20250623,DC,NC,2733438,Other Designation,Lands,https://irma.nps.gov/DataStore/Reference/Profi...,National Mall,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{85419690-57D3-4960-8EC5-54D765A7E841},0.0,1.154398e+06,"POLYGON ((-77.02387 38.89192, -77.02387 38.891..."
1,2,BUFF,Lands - http://landsnet.nps.gov/tractsnet/docu...,Buffalo National River,20250616,AR,MW,74272,National River,Lands,https://irma.nps.gov/DataStore/Reference/Profi...,Buffalo,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{A8834374-D542-4874-BC96-A4FFAF398780},0.0,5.798325e+08,"POLYGON ((-92.41246 36.17096, -92.40795 36.170..."
2,3,GOGA,Lands - http://landsnet.nps.gov/tractsnet/docu...,Golden Gate National Recreation Area,20250616,CA,PW,255952,National Recreation Area,Lands,https://irma.nps.gov/DataStore/Reference/Profi...,Golden Gate,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{50E142BB-3EB2-4E0C-8E3A-D1F8726420B3},0.0,5.356688e+08,"MULTIPOLYGON (((-122.51849 37.54014, -122.5174..."
3,4,FRST,Lands - http://landsnet.nps.gov/tractsnet/docu...,First State National Historical Park,20250702,DE,NE,2756166,National Historical Park,Lands,https://irma.nps.gov/DataStore/Reference/Profi...,First State,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{4037A3D1-6534-4F78-879D-863C9CF2FB9D},0.0,9.679907e+06,"MULTIPOLYGON (((-75.52321 39.15614, -75.52356 ..."
4,5,BEOL,Lands - http://landsnet.nps.gov/tractsnet/docu...,Bent's Old Fort National Historic Site,20250616132417,CO,IM,2559483,National Historic Site,Lands,https://irma.nps.gov/DataStore/Reference/Profi...,Bent's Old Fort,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{39904341-7BBB-419B-947F-AABEAAEF4211},0.0,5.420545e+06,"POLYGON ((-103.41858 38.04994, -103.41853 38.0..."


In [325]:
cols = ["UNIT_CODE", 
        "GIS_Notes", 
        "UNIT_NAME", 
        "STATE", 
        "REGION", 
        "GNIS_ID",
        "UNIT_TYPE",
        "CREATED_BY"]

df_geo[cols] = df_geo[cols].apply(clean)

df_geo.head()

,OBJECTID,UNIT_CODE,GIS_Notes,UNIT_NAME,DATE_EDIT,STATE,REGION,GNIS_ID,UNIT_TYPE,CREATED_BY,METADATA,PARKNAME,CreationDa,Creator,EditDate,Editor,GlobalID,Shape_Leng,Shape_Area,geometry
0,1,mall,lands - http://landsnet.nps.gov/tractsnet/docu...,national mall,20250623,dc,nc,2733438,other designation,lands,https://irma.nps.gov/DataStore/Reference/Profi...,National Mall,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{85419690-57D3-4960-8EC5-54D765A7E841},0.0,1.154398e+06,"POLYGON ((-77.02387 38.89192, -77.02387 38.891..."
1,2,buff,lands - http://landsnet.nps.gov/tractsnet/docu...,buffalo national river,20250616,ar,mw,74272,national river,lands,https://irma.nps.gov/DataStore/Reference/Profi...,Buffalo,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{A8834374-D542-4874-BC96-A4FFAF398780},0.0,5.798325e+08,"POLYGON ((-92.41246 36.17096, -92.40795 36.170..."
2,3,goga,lands - http://landsnet.nps.gov/tractsnet/docu...,golden gate national recreation area,20250616,ca,pw,255952,national recreation area,lands,https://irma.nps.gov/DataStore/Reference/Profi...,Golden Gate,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{50E142BB-3EB2-4E0C-8E3A-D1F8726420B3},0.0,5.356688e+08,"MULTIPOLYGON (((-122.51849 37.54014, -122.5174..."
3,4,frst,lands - http://landsnet.nps.gov/tractsnet/docu...,first state national historical park,20250702,de,ne,2756166,national historical park,lands,https://irma.nps.gov/DataStore/Reference/Profi...,First State,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{4037A3D1-6534-4F78-879D-863C9CF2FB9D},0.0,9.679907e+06,"MULTIPOLYGON (((-75.52321 39.15614, -75.52356 ..."
4,5,beol,lands - http://landsnet.nps.gov/tractsnet/docu...,bent's old fort national historic site,20250616132417,co,im,2559483,national historic site,lands,https://irma.nps.gov/DataStore/Reference/Profi...,Bent's Old Fort,None,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{39904341-7BBB-419B-947F-AABEAAEF4211},0.0,5.420545e+06,"POLYGON ((-103.41858 38.04994, -103.41853 38.0..."


In [326]:
df_geo.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 437 entries, 0 to 436
Data columns (total 20 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   OBJECTID    437 non-null    int64   
 1   UNIT_CODE   437 non-null    object  
 2   GIS_Notes   437 non-null    object  
 3   UNIT_NAME   437 non-null    object  
 4   DATE_EDIT   437 non-null    object  
 5   STATE       436 non-null    object  
 6   REGION      437 non-null    object  
 7   GNIS_ID     428 non-null    object  
 8   UNIT_TYPE   437 non-null    object  
 9   CREATED_BY  437 non-null    object  
 10  METADATA    437 non-null    object  
 11  PARKNAME    437 non-null    object  
 12  CreationDa  0 non-null      object  
 13  Creator     437 non-null    object  
 14  EditDate    437 non-null    object  
 15  Editor      437 non-null    object  
 16  GlobalID    437 non-null    object  
 17  Shape_Leng  414 non-null    float64 
 18  Shape_Area  437 non-null    float64 
 19  

In [327]:
df_geo = df_geo.drop(columns=["CreationDa"])

df_geo.columns

Index(['OBJECTID', 'UNIT_CODE', 'GIS_Notes', 'UNIT_NAME', 'DATE_EDIT', 'STATE',
       'REGION', 'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY', 'METADATA', 'PARKNAME',
       'Creator', 'EditDate', 'Editor', 'GlobalID', 'Shape_Leng', 'Shape_Area',
       'geometry'],
      dtype='object')

I'll save the obtained dataset as a .gpkg file so that I won't lose the geometry.

In [328]:
df_geo.to_file("../Data/NPS_Geo.gpkg", driver="GPKG")

Add park_id using the visits code, which should be a match.

In [329]:
# Add park_id to df_geo using UNIT_CODE
df_geo['park_id'] = df_geo['UNIT_CODE'].str.lower().map(park_id_map)

df_geo.head(1)

,OBJECTID,UNIT_CODE,GIS_Notes,UNIT_NAME,DATE_EDIT,STATE,REGION,GNIS_ID,UNIT_TYPE,CREATED_BY,METADATA,PARKNAME,Creator,EditDate,Editor,GlobalID,Shape_Leng,Shape_Area,geometry,park_id
0,1,mall,lands - http://landsnet.nps.gov/tractsnet/docu...,national mall,20250623,dc,nc,2733438,other designation,lands,https://irma.nps.gov/DataStore/Reference/Profi...,National Mall,NPS_WASO_LANDS,20250703144021,NPS_WASO_LANDS,{85419690-57D3-4960-8EC5-54D765A7E841},0.0,1.154398e+06,"POLYGON ((-77.02387 38.89192, -77.02387 38.891...",NaN


In [330]:
df_geo.columns

Index(['OBJECTID', 'UNIT_CODE', 'GIS_Notes', 'UNIT_NAME', 'DATE_EDIT', 'STATE',
       'REGION', 'GNIS_ID', 'UNIT_TYPE', 'CREATED_BY', 'METADATA', 'PARKNAME',
       'Creator', 'EditDate', 'Editor', 'GlobalID', 'Shape_Leng', 'Shape_Area',
       'geometry', 'park_id'],
      dtype='object')

In [331]:
df_geo = df_geo.drop(columns=["DATE_EDIT", 'GNIS_ID', 'CREATED_BY', 'Creator', 'EditDate', 'Editor', 'GNIS_ID'])

In [332]:
df_geo.columns

Index(['OBJECTID', 'UNIT_CODE', 'GIS_Notes', 'UNIT_NAME', 'STATE', 'REGION',
       'UNIT_TYPE', 'METADATA', 'PARKNAME', 'GlobalID', 'Shape_Leng',
       'Shape_Area', 'geometry', 'park_id'],
      dtype='object')

In [ ]:
df_geo = df_geo.rename(columns={
    "OBJECTID": 'geo_id',
    'UNIT_CODE': 'geo_code',
    'GIS_Notes': 'notes',
    "UNIT_NAME": "geo_name",
    "STATE": "state",
    "REGION": "region",
    "UNIT_TYPE": "unit_type",
    "METADATA": "metadata",
    'PARKNAME': 'park_name',
    'GlobalID': 'global_id',
    "Shape_leng": "shape_length",
    "Shape_Area": "shape_area"
})

df_geo.columns

Index(['geo_id', 'geo_code', 'notes', 'geo_name', 'state', 'region',
       'unit_type', 'metadata', 'park_name', 'global_id', 'Shape_Leng',
       'shape_area', 'geometry', 'park_id'],
      dtype='object')

In [333]:
df_units.head(1)

,units_name,type,units_code,org_code,region,park_id
1,abraham lincoln birthplace,nhp,abli,5540,ser,1.0


Now, I'll create my database and load the relevent dataframes.

In [ ]:
#Connect to db (or create on first run)
conn = sqlite3.connect('../data/NPS_Mortality_Project.db')

# Load all tables
df_park_units.to_sql('park_units', conn, if_exists='replace', index=False)
df_dates.to_sql('dates', conn, if_exists='replace', index=False)
df_cause_types.to_sql('cause_types', conn, if_exists='replace', index=False)
df_intent_types.to_sql('intent_types', conn, if_exists='replace', index=False)
df_sex_types.to_sql('sex_types', conn, if_exists='replace', index=False)
df_activity_types.to_sql('activity_types', conn, if_exists='replace', index=False)
df_amenity_types.to_sql('amenity_types', conn, if_exists='replace', index=False)
df_mortality_incidents.to_sql('mortality_incidents', conn, if_exists='replace', index=False)
df_amenities.to_sql('amenities', conn, if_exists='replace', index=False)
df_park_activities.to_sql('park_activities', conn, if_exists='replace', index=False)
df_visits.to_sql('visits', conn, if_exists='replace', index=False)
#drop geo column for db
df_geo.drop(columns=['geometry']).to_sql('geo', conn, if_exists='replace', index=False)

#Create indexes
cursor = conn.cursor()

cursor.execute("CREATE INDEX IF NOT EXISTS idx_incidents_park ON mortality_incidents(park_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_incidents_date ON mortality_incidents(date_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_incidents_cause ON mortality_incidents(cause_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_amenities_park ON amenities(park_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_amenities_type ON amenities(amenity_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_activities_park ON park_activities(park_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_visits_park ON visits(park_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_geo_park ON geo(park_id)")

conn.commit()

In [ ]:
conn = sqlite3.connect('../data/NPS_Mortality_Project.db')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

Tables in database:
                   name
0            park_units
1                 dates
2           cause_types
3          intent_types
4             sex_types
5        activity_types
6         amenity_types
7   mortality_incidents
8             amenities
9       park_activities
10               visits
11                  geo
